# SmolVLA × LIBERO-plus Spatial Fine-tuning——最终课题教材（基础版）

## 教材定位

本教材用于体验从零开始用 **PyTorch** 编写 **Vision-Language-Action（VLA，视觉-语言-动作）** 模型，并在机器人仿真环境中训练和评估的完整工作流。

我们实现一个吸收 SmolVLA（Hugging Face, 2025）**核心概念**的教育版简化 VLA：冻结 VLM + Flow Matching + action chunking。它不是官方代码的完整复刻，而是优先保证可学习、可检查的最小结构（与官方实现的差异见 section 8.5）：

1. 用预训练权重加载并冻结 **Vision-Language 部分**（SmolVLM2-500M）；
2. 从零实现并训练 **Action 部分**（基于 Flow Matching 的 transformer）；
3. 在 LIBERO-plus 环境中测量**成功率**并输出 rollout **视频**；
4. 把训练条件和结果整理为**提交用 JSON**。

## 🎯 基础版需要提交的内容

**填完所有空缺并从上到下运行 notebook 后，会生成 `submission.json`。**

`section 8-9` 中有 `TODO 1` 到 `TODO 4` 四处**填空题**（用 `raise NotImplementedError(...)` 明确标记）。如果没有全部正确填写，notebook 会在中途停止，无法生成提交 JSON。

### <font color="red">※ 完成课题提交前，请不要编辑填空位置以外的内容</font>

| # | 位置 | 要实现的内容 | 提示（公式） |
|---|---|---|---|
| TODO 1 | 8.1 `SinusoidalTimeEmbedding.forward` | 时间与频率的外积 | $\text{args}[b, i] = t[b] \cdot \text{freqs}[i]$ |
| TODO 2 | 8.5 `SmolVLA.compute_loss` | 线性路径上的中间点 $x_t$ | $x_t = (1-t)\,x_0 + t\,a$ |
| TODO 3 | 8.5 `SmolVLA.compute_loss` | 真实速度 $v^*$ | $v^* = a - x_0$ |
| TODO 4 | 8.5 `SmolVLA.sample_actions` | 一步 Euler 更新 | $x \leftarrow x + v \cdot dt$ |

四个答案都只是**一行短公式**。在 `raise` 后有 `... = ??` 占位符；删除 `raise` 那一行并填写 `??`。

### 提交物（基础版）

执行 `section 11` 会在浏览器下载 **`submission.json`**，同时备份到 Drive：

- **`submission.json`** ← ★唯一需要提交的文件★
- `submission.md` / `pip_freeze.txt` —— 仅供参考，不需提交

基础版的目标是“**实现能运行**”，不是比拼分数。**成功率不用于判定**；教育版从零训练且数据量小，基本以失败为预期（**即使成功率 0% 也合格**）。想竞争数值成绩或展示改进，请使用另一个 Advanced notebook。

## 运行环境（Colab）

本 notebook 以 **Google Colab** 为目标：

- 工作目录为 `/content/workdir`（优先保证运行速度，属于临时目录）；
- 产物会自动复制到 Google Drive 的 `MyDrive/smolvla_task/`，避免运行时结束后丢失。

**运行前必须完成**：

- 选择 **运行时 → 更改运行时类型 → GPU**（T4 或更高）；
- Section 1 请求挂载 Drive 时允许授权。

**Colab 注意事项**：

- LIBERO-plus assets（6GB）的下载与展开需要时间，请提前确认连接稳定；
- Colab 有连续运行时长限制，必要时分段执行并保存进度；
- Drive 剩余空间不足会导致备份失败，请提前检查 `MyDrive`。

## 数据与环境

训练和评估都基于 [LIBERO-plus](https://github.com/sylvestf/LIBERO-plus)。

### 训练数据

- **Hugging Face dataset**：[`lerobot/libero_plus`](https://huggingface.co/datasets/lerobot/libero_plus)
- **任务**：LIBERO-Spatial 的 10 个任务（都是“把黑碗放到盘子上”的变体）
- **episode 数**：10 个任务 × 5 episode = **50 个 episode**（section 7 抽取）

### 评估环境

- **仿真器**：`LiberoEnv`（经由 LeRobot 使用 LIBERO-plus 扰动仿真）
- **任务**：从 LIBERO-Spatial 10 个任务中取 2 个任务各 1 个 episode（共 2 次 rollout）

### 数据流

| 步骤 | 内容 | 来源 |
|---|---|---|
| 抽取数据 | 从 Hugging Face `lerobot/libero_plus` 选出 50 个 episode | section 7 |
| 训练 | 从零训练 PyTorch SmolVLA | section 9 |
| 评估 | 在 `LiberoEnv` 中 rollout | section 10 |
| 提交 | **生成 `submission.json` 并下载** | section 11 |

## 默认条件（可在 `section 6` 修改）

- 训练 episode：LIBERO-Spatial 10 个任务 × 每个 5 个；
- 训练 step：300；
- 模型：`transformer_hidden_dim=256`、4 层（可在 `section 8.5` 调整，同时列出论文规模）；
- 评估：2 个任务 × 1 个 episode = 2 次 rollout（自定义 SmolVLA）。


## 📖 目录与各 section 的作用

按从上到下的顺序执行。带 **★** 的部分是本课题的核心。

### 【准备阶段】环境与数据

| # | Section | 内容 |
|---|---|---|
| 1 | Colab runtime 检查 | 检查 GPU / Python 版本并设置 `WORKDIR` |
| 2 | 系统包 | 用 apt 安装 `ffmpeg` / `libgl` 等 |
| 3 | LeRobot 安装 | clone v0.6.0 并 editable install |
| 4 | HF 获取辅助函数 | 定义 429 重试和缓存优先的获取函数 |
| 5 | LIBERO-plus 环境 | pin 安装 MuJoCo + robosuite，并下载/展开 6GB assets |
| 6 | 设置训练/评估条件 | 定义数据集、seed、路径等共享常量 |
| 7 | Spatial 数据抽取 | 等间隔抽取 10 个任务 × 5 episode = 50 个 |

### 【★核心】实现 SmolVLA → 训练 → 评估

| # | Section | 内容 |
|---|---|---|
| **8** | **★ 用 PyTorch 实现 SmolVLA** | **包含 TODO 1–4** |
| 8.1 | 时间嵌入 | 把连续时间 `t` 变成高维向量（**TODO 1**） |
| 8.2 | Transformer block | Self/Cross attention + MLP |
| 8.3 | Flow Matching Head | 预测速度场的 transformer |
| 8.4 | 加载并冻结 SmolVLM2 | 以 fp16/bf16 加载并冻结 500M VLM |
| 8.5 | 组装 SmolVLA | 连接 VLM 与 Action head（**TODO 2、3、4**） |
| 8.6 | 检查可训练参数 | 打印 trainable / frozen 的构成 |
| **9** | **训练** | 用 50 个 episode 训练自定义 SmolVLA |
| 9.1 | LIBERO 数据集 | 用 `LeRobotDataset` 提供带 action chunk 的样本 |
| 9.2 | action 归一化 | 计算全部 action 的 mean/std |
| 9.3 | PyTorch 训练循环 | AdamW + warmup+cosine + 梯度裁剪 |
| **10** | **评估与 rollout 视频** | 在 LIBERO 环境中运行并生成 mp4 |
| 10.1 | LIBERO rollout | 2 个任务 × 1 episode 测成功率并收集 frame |
| 10.2 | 制作视频 | 用 `av`（PyAV）制作 mp4 并嵌入 notebook |
| **11** | **★ 导出提交日志** | **这里生成 `submission.json`** |

Advanced 课题（LeRobot 路线、25 个任务的广域评估）在单独的 `smolvla_libero_plus_spatial_advanced.ipynb` 中完成。

### 💡 注意

### <font color="red">※ Section 1–7 是环境和数据准备，按要求执行即可</font>
### <font color="red">※ 填写 Section 8 的 TODO，然后执行 9–11</font>
### <font color="red">※ 提交完成前不要编辑填空位置以外的内容</font>


## 1. 检查 Colab runtime

请先把 Colab runtime 切换到 GPU，再执行此 cell。


In [ ]:
# ==============================================================
# 环境变量与运行前检查 + Google Drive 挂载
# ==============================================================
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

# --- Google Drive 挂载 (持久化产物) --------------------
# Colab 的 /content/ 运行时结束后该目录会被清除，因此、submission.json 和
# 训练好的模型可以 Drive 保存到 Drive，因此先挂载.
# 会出现认证弹窗，请在浏览器中允许 (首下一次).
from google.colab import drive

drive.mount("/content/drive")

# Drive 复制产物到 Drive 的位置. `MyDrive/smolvla_task/` 配下.
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/smolvla_task")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"DRIVE_BACKUP_DIR: {DRIVE_BACKUP_DIR}")

# --- 工作目录 ------------------------------------------
# 为优先运行速度，使用 /content/ (ephemeral) 到置. Drive 直接在 Drive 上操作会使
# 45 万个小文条的展开变慢. 仅最终产物 Drive 复制到其中.
WORKDIR = Path("/content/workdir")
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"WORKDIR         : {WORKDIR}")

# --- Hugging Face Hub 的抑制噪声 ------------------------------
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
# HF / LeRobot 的缓存固定到 WORKDIR 目录下
os.environ["HF_HOME"] = str(WORKDIR / "hf_cache")
os.environ["HF_LEROBOT_HOME"] = str(WORKDIR / "lerobot_cache")
# 长时间训练中容易出现的碎片化 OOM 以避免
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# LeRobot v0.6.0 是 Python 3.12 以上是必需的
if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上是必需的。")

# GPU 必須
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU请选择 GPU runtime "
        "(Colab 菜单: runtime → 更改 runtime 类型 → GPU)"
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 准备系统包

安装 LeRobot、视频解码和 MuJoCo 所需的系统包。


In [ ]:
# ==============================================================
# subprocess 执行辅助函几 + apt 并安装系列列统包
# ==============================================================


# 外部命令， stdout/stderr 统一捕获后执行的辅助函几:
#   - 成功时不输出日志， notebook 避免污染 notebook
#   - 失败时只把末尾 6000 个字符放入异常以便查看
#   - stderr 是 stdout 合并到 stdout 以保留顺序
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


# 用于长时间命令: 1 每收到一行 print 执行 streaming 版个个。
# 看不到进度 = 避免被误认为无限循环的辅助函几。
# - prefix 加上前缀以标明命令输出
# - 最近的日志 200 行保存在环形缓冲区中，失败时用于转储
def run_streaming(
    command: list[str],
    *,
    prefix: str = "",
    check: bool = True,
) -> int:
    from collections import deque

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    recent: deque[str] = deque(maxlen=200)
    assert process.stdout is not None
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        print(f"{prefix}{line}", flush=True)
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# 用于长时间命令 (pip install 等输出达到几百行的情况):
# 直近 tail 仅最近几行 ipywidgets 的 HTML box ， "上書更新" 显示执行.
# 显示完整时间线会使 notebook 膨胀，因此、只显示末尾.
def run_streaming_rolling(
    command: list[str],
    *,
    prefix: str = "",
    tail: int = 10,
    check: bool = True,
) -> int:
    from collections import deque
    import ipywidgets as widgets
    from IPython.display import display

    recent: deque[str] = deque(maxlen=tail)
    _widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_widget)

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert process.stdout is not None
    _count = 0
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        _count += 1
        # 逐行更新会使 UI 变慢，因此 5 每若干行更新 widget
        if _count % 5 == 0:
            _widget.value = (
                "<pre style='margin:0;font-size:11px;color:#666;"
                "line-height:1.3'>"
                + "\n".join(f"  {prefix}{l}" for l in recent)
                + "</pre>"
            )
    # 反映到最后一行
    _widget.value = (
        "<pre style='margin:0;font-size:11px;color:#666;line-height:1.3'>"
        + "\n".join(f"  {prefix}{l}" for l in recent)
        + "\n  ✅ done ({_count} lines total)".replace("{_count}", str(_count))
        + "</pre>"
    )
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# apt 缓存更新
run_quiet(["sudo", "apt-get", "update", "-qq"])
# 依赖包说明:
#   ffmpeg                : LeRobot 的视频解码
#   git / unzip           : 个 notebook 中 clone / 用于展开
#   libgl1 / libglib2.0-0 : OpenGL 系列列（MuJoCo/robosuite 的渲染）
#   libsm6 / libxext6     : X11 系列列列 runtime（渲染库依赖）
#   libexpat1             : XML 解析器（MuJoCo 的 XML 模型加载）
#   libfontconfig1-dev    : matplotlib 用字体设置
#   libmagickwand-dev     : Wand（ImageMagick binding） LIBERO 经由它需要
run_quiet(
    [
        "sudo", "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "git",
        "unzip",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libexpat1",
        "libfontconfig1-dev",
        "libmagickwand-dev",
    ]
)

print("System packages ready.")

## 3. 安装 LeRobot

使用 LeRobot `v0.6.0`，本 cell 同时应用 Colab LoRA 训练所需的兼容性调整。


In [ ]:
# ==============================================================
# LeRobot v0.6.0 的 editable install 与兼容性补丁
# ==============================================================
# 手順:
#   1. 现有 install 的除去（torchao 也同时。SmolVLA 因为不支持）
#   2. 固定 tag 的 LeRobot  shallow clone（进度 stream）
#   3. bf16 不支持 GPU 若 SmolVLM  fp16 改写为
#   4. lerobot_train.py 的冗长日志和 tqdm 用轻量补丁抑制
#   5. editable install（training + smolvla + peft 的 extras）— 输出 stream
#   6. torchao 重新次卸载（防止依赖重新安装的保险措施）
#   7. sys.modules 与 sys.path 掃除并 clone 完成 src 设为优先
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = WORKDIR / "lerobot"
LEROBOT_SRC = LEROBOT_DIR / "src"

# --- 0. 安全部部保护: torch 的版个个个一致性检查 ------------
# 上一个 session 中 torch  import 已导入，之后 pip install 
# torch 更新后、C 扩展(.so)是旧的的版个个，・Python 侧为新版个个的
# 不一致状态会导致 "cannot import name '_EvalFrameOverride'" 等问题。
# 事先比较磁盘上的版个个个与内存中的 in-memory 的 torch 版个个个、
# 若不同则提示重启 kernel。
# torch.__version__ / importlib.metadata.version("torch") 在不同环境中可能
# '2.11.0+cu128' 的到 CUDA 标签 (local version identifier) 可能带有
# 也可能没有. 从两侧都去除 '+' 之后的内容 PEP 440 的 public version 后重新比较.
def _strip_local_version(v: str) -> str:
    return v.split("+", 1)[0]

_torch_version_in_memory = _strip_local_version(torch.__version__)
try:
    _torch_dist_version = _strip_local_version(
        importlib.metadata.version("torch")
    )
except importlib.metadata.PackageNotFoundError:
    _torch_dist_version = None

if _torch_dist_version and _torch_dist_version != _torch_version_in_memory:
    raise RuntimeError(
        f"torch mismatch: in-memory={_torch_version_in_memory} "
        f"on-disk={_torch_dist_version}. "
        "kernel重新起動并的cell最初从执行直并请。"
    )

# --- 1. 现有 install 移除（为了安全部部重复执行） ----------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

# --- 2. LeRobot  shallow clone（进度 stream 显示） -----------
# 网络慢时需要几分钟，但、输出很多，所以只显示最近 10 行
print("Cloning LeRobot...", flush=True)
run_streaming_rolling(
    [
        "git",
        "clone",
        "--progress",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ],
    prefix="[git] ",
    tail=10,
)

# --- 3. dtype 兼容性补丁 -------------------------------------
# T4/V100 系列列等 bf16 不支持 GPU ，是 bf16 ， load 会导致损坏，因此 fp16 替换为
smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

# --- 4. lerobot_train.py 的軽微补丁 ---------------------------
# 训练 subprocess 的压缩日志:
#   - 配置 pformat 的 INFO  DEBUG 到降低
#   - inside_slurm() 不论判定结果 tqdm 强制 disable
train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

# --- 5. editable install（extras 包含） ------------------------
# --upgrade 不添加: torch 如果包含它的依赖已经满足，
# 就不改动任何内容（现有的 C 以避免与 C 扩展不一致）。
# 解析依赖和构建需要 5〜20 分钟. pip 的输出是几百行会变成的， rolling 显示 (直近 10 行仅).
print("Installing LeRobot (this may take a while)...", flush=True)
run_streaming_rolling(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ],
    prefix="[pip] ",
    tail=10,
)

# --- 5.5. 安装后 torch 重新次确认运行正常 --------
_torch_dist_version_after = _strip_local_version(
    importlib.metadata.version("torch")
)
if _torch_dist_version_after != _torch_version_in_memory:
    raise RuntimeError(
        f"pip install  torch  {_torch_version_in_memory} → "
        f"{_torch_dist_version_after} 已从旧的版个个变更为新版个个。"
        "kernel重新起動并的cell最初从执行直并请。"
    )

# --- 6. torchao 重新卸载 ------------------------------
# 上的 install 可能在安装中作为依赖恢复，因此重新次删除
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

# --- 7. sys.modules 与 sys.path 的掃除 --------------------------
# 上次执行 cell 时加载的旧的的 lerobot/torchao 模块移出缓存
for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

# clone 完成 src 以优先解析 sys.path 替换
sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

# torchao 检查是否完全部部删除。如果还存在， SmolVLA 会在启动时失败
try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchao的删除到失败完成。")

# 实际 import 并、read 路径位于当前 clone 完成 src 目录下
# 首次 import 是 torch/CUDA 初始化可能需要 30〜60 秒
print("Importing lerobot (torch init may take up to a minute)...", flush=True)
import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobot的加载路径不正确。")

print("LeRobot ready.")

## 4. 准备公开文件获取流程

优先使用缓存；匿名访问受限时自动重试。


In [ ]:
# ==============================================================
# Hugging Face 获取重试 + 缓存优先 wrapper
# ==============================================================
# 匿名称称访问容易触发 429 (Too Many Requests) ，因此实现指几退避
# 、若已有缓存则跳过实际 DL 以避免。
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


# 429 最多 6 次重试。其他错误立即重新抛出。
def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            # 429 以外不重试（404 不反复请求）
            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            # 如果服务器返次 Retry-After 则遵守它、否则使用指几退避
            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Face获取失败。"
    ) from last_error


# 先 local_files_only 检查缓存、若没有则实际 DL 执行。
# 大型模型/避免重复获取几据集的优化。
def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        # 命中缓存时不访问网络并立即返次
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        # 缓存未命中 → 429 带重试实际 DL
        # max_workers=1 以温和方式访问服务器（429 次避）
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 5. 准备 LIBERO-plus 评估环境

安装 MuJoCo / robosuite / LIBERO-plus fork，并展开评估所需 assets。

由于需要下载并展开约 6GB 的 `assets.zip`，这个 cell 会花较长时间。


In [ ]:
# ==============================================================
# LIBERO-plus 设置评估环境
# ==============================================================
# 手順:
#   1. MuJoCo/robosuite 系列列的依赖 pin 带有 install
#   2. LIBERO-plus fork 固定 SHA ， clone/checkout
#   3. assets.zip  HF 从其获取并展开
#   4. ~/.libero/config.yaml 書asset位置教
#   5. lerobot_eval.py 到进度显示用的补丁当
#   6. sys.path / sys.modules 掃除并 fork 側优先加载
from huggingface_hub import hf_hub_download
import time

LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_DIR = WORKDIR / "LIBERO-plus"
LIBERO_PLUS_PACKAGE_ROOT = (
    LIBERO_PLUS_DIR / "libero" / "libero"
)
# assets (45 万小文条 6GB) 的保存先. /content/ 配下因此展开使用高速.
LIBERO_PLUS_ASSETS_DIR = WORKDIR / "libero_plus_assets" / "assets"

# Colab 的无头 GPU 环境，是 EGL 后端， MuJoCo 描画
os.environ["MUJOCO_GL"] = "egl"

# --- 1. 现有 install 移除（重新执行安全部部性） --------------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "hf-libero",
        "libero",
        "robosuite",
    ],
    check=False,
)

# --- 2. LIBERO-plus 依赖的 pin 付 install --------------------
# -揃与 robosuite / mujoco 的 API 差，崩溃执行为了厳格到固定
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "robosuite==1.4.1",  # LIBERO-plus 预期执行固定版个个
        "bddl==1.0.1",
        "easydict==1.13",
        "mujoco==3.7.0",
        "matplotlib==3.10.8",
        "Wand==0.6.13",
        "scikit-image==0.25.2",
        "gym==0.26.2",
        "future",  # bddl 1.0.1 依赖宣言并隠依赖
    ]
)

# robosuite 预期的版个个加载被与担保
if (
    importlib.metadata.version("robosuite")
    != "1.4.1"
):
    raise RuntimeError(
        "robosuite 1.4.1 is required."
    )

# --- 3. LIBERO-plus fork  clone -----------------------------
if not (LIBERO_PLUS_DIR / ".git").is_dir():
    shutil.rmtree(
        LIBERO_PLUS_DIR,
        ignore_errors=True,
    )
    run_quiet(
        [
            "git",
            "clone",
            "--quiet",
            "https://github.com/sylvestf/LIBERO-plus.git",
            str(LIBERO_PLUS_DIR),
        ]
    )

# 固定 SHA 方向 checkout。现有 clone 到対并也重新执行安全部部会变成、
# checkout 失败時仅 fetch ，恢复执行
checkout = run_quiet(
    [
        "git",
        "-C",
        str(LIBERO_PLUS_DIR),
        "checkout",
        "--quiet",
        LIBERO_PLUS_SHA,
    ],
    check=False,
)

if checkout.returncode != 0:
    # SHA -课程到没有（shallow clone 等）若 fetch 并从 checkout
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "fetch",
            "--quiet",
            "--depth",
            "1",
            "origin",
            LIBERO_PLUS_SHA,
        ]
    )
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "checkout",
            "--quiet",
            LIBERO_PLUS_SHA,
        ]
    )

# fork  editable install。--no-deps ， pin 済的依赖覆盖更新被
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "-e",
        str(LIBERO_PLUS_DIR),
    ]
)

# --- 4. assets.zip  HF 从 DL・展开 -------------------------
# 大 asset 是 fork 側到含的， Sylvest/LIBERO-plus 从获取
if not LIBERO_PLUS_ASSETS_DIR.is_dir():
    assets_root = WORKDIR / "libero_plus_assets"
    assets_root.mkdir(parents=True, exist_ok=True)

    # ---- Drive 缓存策略 ----
    # 优先级: (1) tar.gz 缓存 (最快)  > (2) zip 缓存  > (3) HF DL
    # tar.gz  Drive 保存后、下次 session 从恢复到展开只需几分钟.
    _drive_tar = DRIVE_BACKUP_DIR / "libero_plus_assets.tar.gz"
    _drive_zip = DRIVE_BACKUP_DIR / "libero_plus_assets.zip"

    if _drive_tar.is_file():
        # ---- 最快路径: tar.gz  Drive 从 Drive 取出并展开 ----
        print(
            f"Drive 的 tar.gz 正在从缓存恢复 (unzip 无需 unzip): {_drive_tar}",
            flush=True,
        )
        _t0 = time.time()
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar

        # tar.gz 是 zip 不同，它没有中央目录，因此、
        # 通常需要流式扫描才能知道文条总几.
        # 因此使用上次的 tar 创建时 .count 把几量写入 sidecar、
        # 如果存在就用它 tqdm 的 total 立即填充 total.
        _drive_tar_count = _drive_tar.with_suffix(_drive_tar.suffix + ".count")
        _tar_total = None
        if _drive_tar_count.is_file():
            try:
                _tar_total = int(_drive_tar_count.read_text().strip())
                print(f"sidecar从 sidecar 获取几量: {_tar_total} files", flush=True)
            except Exception:
                _tar_total = None
        if _tar_total is None:
            # 无 sidecar (旧的的缓存): getmembers() ， 1 扫描一次.
            # data 不展开几据，所以比完整展开快，但、仍需等待几十秒.
            print("正在统计文条几 (首下一次、需要一点时间)...", flush=True)
            with tarfile.open(_drive_tar, "r:gz") as _tf:
                _tar_total = sum(1 for _ in _tf)

        with tarfile.open(_drive_tar, "r:gz") as _tf:
            _pbar_tx = _tqdm_tar(total=_tar_total, desc="tar 展开", unit="files")
            while True:
                _m = _tf.next()
                if _m is None:
                    break
                _tf.extract(_m, assets_root)
                _pbar_tx.update(1)
            _pbar_tx.close()
        print(f"tar 展开完成 (in {time.time()-_t0:.0f}s)", flush=True)

        # 旧的的 zip 缓存固定到删除 (Drive 容量节省)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"旧的 zip 缓存固定到删除: {_drive_zip}", flush=True)
            except Exception:
                pass
        # → LIBERO_PLUS_ASSETS_DIR 应当存在. 以下 DL/unzip 跳过.
        _skip_dl_unzip = True
    else:
        _skip_dl_unzip = False

if not LIBERO_PLUS_ASSETS_DIR.is_dir() and not _skip_dl_unzip:
    # ---- 中速路径: zip 缓存从恢复 or HF DL ----
    local_zip = assets_root / "assets.zip"

    if _drive_zip.is_file() and not local_zip.is_file():
        print(f"Drive 缓存从 assets.zip 恢复: {_drive_zip}", flush=True)
        shutil.copy2(_drive_zip, local_zip)

    if local_zip.is_file():
        archive_path = local_zip
        print(f"现有的 assets.zip 使用: {archive_path}", flush=True)
    else:
        print("HF 从 assets.zip 正在下载 (6GB, 需要时间)...", flush=True)
        # ---- HF DL 期间 progress bar 启用 ----
        # env-var 的 pop 是 huggingface_hub  import 时 disabled 因为被缓存所以不起作用.
        # 官方文档的 API enable_progress_bars() / disable_progress_bars() 显式切换.
        from huggingface_hub.utils import (
            enable_progress_bars,
            disable_progress_bars,
        )
        enable_progress_bars()
        try:
            _t0 = time.time()
            archive_path = Path(
                run_hf_with_retry(
                    lambda: hf_hub_download(
                        repo_id="Sylvest/LIBERO-plus",
                        repo_type="dataset",
                        filename="assets.zip",
                        local_dir=assets_root,
                        token=False,
                    )
                )
            )
            _dl_sec = time.time() - _t0
        finally:
            disable_progress_bars()

        _size_mb = archive_path.stat().st_size / 1e6
        print(
            f"HF DL 完成: {_size_mb:.0f} MB in {_dl_sec:.0f}s "
            f"({_size_mb / max(_dl_sec, 1):.1f} MB/s)",
            flush=True,
        )
        # 注意意: 在此阶段 Drive 到 zip 書还是.
        # 展开后到 tar.gz 创建 Drive 到置 (下面的处理) 更
        # 也能更快、更省空间地从下次 session 恢复.
    extract_dir = assets_root / "extract"

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    # 展开使用 Python 的 zipfile + tqdm progress bar 执行.
    # ・整体进度用 tqdm 进度条显示 (ETA / files/sec 包含).
    # ・直近 10 文条名称称 ipywidgets 的 HTML box ， "上書更新" 执行
    #   (subprocess 的 unzip -o 会导致 45 万行输出并让 notebook 卡死，因此).
    import zipfile
    import ipywidgets as widgets
    from IPython.display import display
    from tqdm.auto import tqdm
    from collections import deque

    print(f"Extracting {archive_path.name} to {extract_dir}...", flush=True)

    # 最近展开的文条 10 个文条的 widget
    _recent_paths: deque[str] = deque(maxlen=10)
    _recent_widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_recent_widget)

    _t0 = time.time()
    with zipfile.ZipFile(archive_path) as _zf:
        _members = _zf.infolist()
        _pbar = tqdm(_members, desc="unzip", unit="files")
        for _i, _m in enumerate(_pbar):
            _recent_paths.append(_m.filename)
            _zf.extract(_m, extract_dir)
            # 500 每个文条 widget 覆盖更新 (每次更新会很慢)
            if _i % 500 == 0:
                _recent_widget.value = (
                    "<pre style='margin:0;font-size:11px;color:#666;"
                    "line-height:1.3'>"
                    + "\n".join(f"  {p}" for p in _recent_paths)
                    + "</pre>"
                )
        # 最终状态
        _recent_widget.value = (
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>"
            + f"  ✅ done: {len(_members)} files extracted"
            + "</pre>"
        )
    print(f"unzip 完成 (in {time.time()-_t0:.0f}s)", flush=True)

    # zip 中的目录层级可能变化，因此 rglob ， assets/ 搜索。
    # 优先选择嵌套较浅的路径（真实的 assets 预计更接近根目录）
    candidates = sorted(
        [
            path
            for path in extract_dir.rglob("assets")
            if path.is_dir()
        ],
        key=lambda path: len(path.parts),
    )

    if not candidates:
        raise FileNotFoundError(
            "LIBERO-plus assets not found."
        )

    LIBERO_PLUS_ASSETS_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    # 展开后的真实 assets  fork 移动到 fork 内的预期位置
    shutil.move(
        str(candidates[0]),
        str(LIBERO_PLUS_ASSETS_DIR),
    )
    # assets 移动完成，因此 extract 只删除临时目录。
    shutil.rmtree(extract_dir, ignore_errors=True)

    # ---- 为下次 session tar.gz  Drive 缓存到 ----
    # zip + unzip 的完整流程很慢，因此、展开后的 assets  tar.gz 化并 Drive 保存到.
    # 下次 session 中 tar 只需展开、几分钟即可恢复.
    try:
        _t0 = time.time()
        print(
            f"Drive 到 tar.gz 正在创建缓存...",
            flush=True,
        )
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar_c

        # 创建时可以预先统计文条几，因此 tqdm 的 total 可以传入 total
        _entries = list(LIBERO_PLUS_ASSETS_DIR.rglob("*"))
        _total_entries = len(_entries) + 1  # +1: root dir 个身
        _pbar_tc = _tqdm_tar_c(total=_total_entries, desc="tar 创建", unit="files")

        def _tar_filter(tarinfo):
            _pbar_tc.update(1)
            return tarinfo

        with tarfile.open(_drive_tar, "w:gz", compresslevel=1) as _tf:
            _tf.add(LIBERO_PLUS_ASSETS_DIR, arcname="assets", filter=_tar_filter)
        _pbar_tc.close()
        # 下次恢复时 tqdm 的 total 以便立即显示、保存文条几 sidecar.
        _drive_tar.with_suffix(_drive_tar.suffix + ".count").write_text(
            str(_total_entries) + "\n"
        )
        print(f"tar.gz 缓存完成 (in {time.time()-_t0:.0f}s)", flush=True)
        # 旧的的 zip 缓存固定到删除 (Drive 容量节省)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"旧的 zip 缓存固定到删除: {_drive_zip}", flush=True)
            except Exception:
                pass
    except Exception as _e:
        print(f"tar.gz 创建缓存失败 (忽略并继续): {_e}", flush=True)

# --- 5. ~/.libero/config.yaml 写入 --------------------------
# LIBERO 启动时读取此文条并解析 assets/bddl/datasets/init_files 的
# 位置。fork 指向 fork 侧路径
# ---- LIBERO 所期望的 assets 路径的 symlink 创建 ----
# LIBERO 的 robosuite Arena XML 加载使用 fork 内的相对路径
# ({LIBERO_PLUS_DIR}/libero/libero/assets/...) 硬编码引用.
# 另一方面，个 notebook 把 notebook 是 45 万个小文条实体放在 LIBERO_PLUS_ASSETS_DIR 中.
# 两者通过 symlink 连接后、LIBERO 是 "预期位置" 即可找到实体.
_libero_expected_assets = LIBERO_PLUS_DIR / "libero" / "libero" / "assets"
if _libero_expected_assets.is_symlink() or _libero_expected_assets.exists():
    if _libero_expected_assets.is_symlink():
        _libero_expected_assets.unlink()
    else:
        shutil.rmtree(_libero_expected_assets, ignore_errors=True)
_libero_expected_assets.parent.mkdir(parents=True, exist_ok=True)
_libero_expected_assets.symlink_to(LIBERO_PLUS_ASSETS_DIR)
print(f"symlink: {_libero_expected_assets} -> {LIBERO_PLUS_ASSETS_DIR}")

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(
    parents=True,
    exist_ok=True,
)
(libero_config_dir / "config.yaml").write_text(
    "\n".join(
        [
            f"assets: {LIBERO_PLUS_ASSETS_DIR}",
            (
                "bddl_files: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}"
            ),
            (
                "datasets: "
                f"{LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}"
            ),
            (
                "init_states: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}"
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)

# --- 6. lerobot_eval.py 应用进度显示补丁 -----------------
# 默认评估脚个会输出冗长的 log 与 tqdm ，所以抑制它。
# 改为 "EVAL_PROGRESS task=x/y episode=a/b" 格式 stdout 写到 stdout、
# 由个 notebook 解析并显示。
eval_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_eval.py"
)
source = eval_script.read_text(encoding="utf-8")

# 冗长的配置 pformat  DEBUG 到降低
source = source.replace(
    "logging.info(pformat(asdict(cfg)))",
    "logging.debug(pformat(asdict(cfg)))",
    1,
)
# 在非录制时 rollout 视频最大 10 个 rollout 视频的部分固定为 0 到固定
source = source.replace(
    "max_episodes_rendered = 0 if cfg.eval.recording else 10",
    "max_episodes_rendered = 0",
    1,
)
# tqdm 强制禁用
source = source.replace(
    "disable=inside_slurm()",
    "disable=True",
)

# 当前 task/episode 把保存进度的全部部局字典注意意入脚个
progress_state = (
    '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
)
if progress_state not in source:
    import_anchor = "from tqdm import trange\n"
    if import_anchor not in source:
        raise RuntimeError(
            "Evaluation progress import anchor not found."
        )
    source = source.replace(
        import_anchor,
        import_anchor + "\n" + progress_state + "\n",
        1,
    )

# task 在循环开头 _EVAL_PROGRESS 插入更新代码
task_loop_anchor = (
    "        for i, (task_group, task_id, env) "
    "in enumerate(tasks):\n"
)
task_loop_patch = (
    task_loop_anchor
    + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
    + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n'
)
if (
    '_EVAL_PROGRESS["task_index"] = i + 1'
    not in source
):
    if task_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation task-loop anchor not found."
        )
    source = source.replace(
        task_loop_anchor,
        task_loop_patch,
        1,
    )

# episode 在循环每轮输出进度 1 行 stdout 写出到 stdout
episode_loop_anchor = "    for batch_ix in progbar:\n"
episode_progress_line = (
    '        print('
    'f"EVAL_PROGRESS '
    "task={_EVAL_PROGRESS['task_index']}/"
    "{_EVAL_PROGRESS['task_total']} "
    'episode={batch_ix + 1}/{n_batches}", '
    "flush=True)\n"
)
if "EVAL_PROGRESS task=" not in source:
    if episode_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation episode-loop anchor not found."
        )
    source = source.replace(
        episode_loop_anchor,
        episode_loop_anchor + episode_progress_line,
        1,
    )

eval_script.write_text(
    source,
    encoding="utf-8",
)

# --- 7. sys.path 清理并验证 fork 的加载 ---------------------
# pip install 完成 LIBERO-plus fork 确保绝对优先
libero_plus_path = str(LIBERO_PLUS_DIR)
sys.path = [
    item
    for item in sys.path
    if item != libero_plus_path
]
sys.path.insert(0, libero_plus_path)

# 旧的的 LIBERO/robosuite 清除模块缓存
for module_name in list(sys.modules):
    if (
        module_name == "libero"
        or module_name.startswith("libero.")
        or module_name == "robosuite"
        or module_name.startswith("robosuite.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

import libero
from libero.libero import benchmark

# libero 包的加载路径位于 fork 目录下
search_paths = [
    Path(path).resolve()
    for path in getattr(libero, "__path__", [])
]

if not any(
    LIBERO_PLUS_DIR.resolve() in path.parents
    or path == LIBERO_PLUS_DIR.resolve()
    for path in search_paths
):
    raise RuntimeError(
        "LIBERO-plus fork was not loaded."
    )

# benchmark 模块也做同样检查
benchmark_path = Path(
    benchmark.__file__
).resolve()

if (
    LIBERO_PLUS_DIR.resolve()
    not in benchmark_path.parents
):
    raise RuntimeError(
        "LIBERO-plus benchmark was not loaded."
    )

print("LIBERO-plus ready.")

## 6. 设置训练与评估条件

这里仅定义**两条路线共有的内容**：

- 模型 / 数据集 repo 与 revision；
- LIBERO-Spatial 的 10 个任务列表；
- 训练使用的 episode 数和 seed；
- 输出路径与混合精度。

每条路线自己的超参数在下面定义：
- PyTorch 版 → `section 9.3` 的 `TRAIN_STEPS_PY` 等；
- LeRobot 版 → Advanced 的 `STEPS` 等。


In [ ]:
# ==============================================================
# 训练・集中定义评估使用的常量和路径
# ==============================================================

# --- 模型 / 固定几据集 revision -----------------------
# revision  commit hash 固定以保证可复现
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = (
    "7bb70aa5bc92b82c9239142775d3a173103567ff"
)

# SmolVLA 的 Vision-Language backbone（500M 参几版个个个）
VLM_REPO = (
    "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
)

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = (
    "f3f49f426d75030177b18778374005bc12ccd588"
)

# --- LIBERO-Spatial 的 10 任务（训练・评估共用） -------------
# 几据集側的 task 名称称称必须严格一致
SPATIAL_TASK_NAMES = [
    "pick up the black bowl from table center and place it on the plate",
    "pick up the black bowl next to the cookie box and place it on the plate",
    "pick up the black bowl next to the plate and place it on the plate",
    "pick up the black bowl next to the ramekin and place it on the plate",
    "pick up the black bowl on the cookie box and place it on the plate",
    "pick up the black bowl on the ramekin and place it on the plate",
    "pick up the black bowl on the stove and place it on the plate",
    "pick up the black bowl on the wooden cabinet and place it on the plate",
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the black bowl between the plate and the ramekin and place it on the plate",
]

# --- 训练・选择评估几据 --------------------------------------
TRAIN_EPISODES_PER_TASK = 5   # 10 tasks × 5 = 50 episodes 选择
SEED = 42                     # 随机 seed
EVAL_SEED = 2026              # 评估 seed

# --- 输出路径 --------------------------------------------------
OUTPUT_DIR = WORKDIR / "outputs" / "smolvla_libero_plus_spatial_lora"
MERGED_MODEL_DIR = WORKDIR / "smolvla_libero_plus_spatial_lora_merged"
BASELINE_MODEL_DIR = WORKDIR / "smolvla_libero_plus_baseline"

BASE_EVAL_DIR = WORKDIR / "eval" / "base"
FINETUNED_EVAL_DIR = WORKDIR / "eval" / "spatial_lora"
COMPARISON_CSV_PATH = WORKDIR / "libero_spatial_comparison.csv"
MERGED_ZIP_PATH = WORKDIR / "smolvla_libero_plus_spatial_lora_merged.zip"

# --- 混合精度模式 --------------------------------------------
# bf16 支持 GPU（A100/H100/L40/RTX40 系列列等）若 bf16、否则 fp16
MIXED_PRECISION = (
    "bf16"
    if torch.cuda.is_bf16_supported()
    else "fp16"
)

## 7. 选择 Spatial 训练数据

从 10 个任务中为每个任务等间隔选择 5 个 episode。


In [ ]:
# ==============================================================
# LIBERO-Spatial 10 从每个任务选择 5 episode 等间隔选择
# ==============================================================
# 可复现地构建训练子集。
# 几据集側的 task 名称称称可能有变体，因此用规范化 key 匹配、
# 每个 task 的 episode index 从等间隔到 5 个样个（合计 50 episode）。
import re
from collections import defaultdict

from lerobot.datasets.dataset_metadata import (
    LeRobotDatasetMetadata,
)


# 转小写・移除符号・规范化空白以生成匹配 key。
def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


# 元几据的 tasks 列可能是 str/list 两者都可能，因此转成单一 str 到降低。
def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value

    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass

    return str(value)


# episode 从集合中等间隔选择 count 个（始末两端都包含）。
def choose_evenly_spaced(
    episode_indices: list[int],
    count: int,
) -> list[int]:
    if len(episode_indices) < count:
        raise ValueError(
            f"Task 中只有 episode  {len(episode_indices)} 条，因此、"
            f"要求几量 {count} 无法满足。"
        )
    positions = [
        round(
            index
            * (len(episode_indices) - 1)
            / (count - 1)
        )
        for index in range(count)
    ]

    return [
        episode_indices[position]
        for position in positions
    ]


# 获取几据集元几据（429 带重试）
dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(
        DATASET_REPO,
        revision=DATASET_REVISION,
    )
)

# 「task 名称称 → 该 task 所属的 episode index 列表」构建
task_to_episodes: dict[str, list[int]] = defaultdict(list)

for episode_index, task_cell in enumerate(
    dataset_metadata.episodes["tasks"]
):
    task_to_episodes[
        task_name_from_cell(task_cell)
    ].append(int(episode_index))

# 归一化-从実際的任务名称称（包含名称称称变体）用于反查的字典
available_by_normalized = {
    normalize_task_name(task_name): task_name
    for task_name in task_to_episodes
}

# Spatial 的 10 从每个任务 5 episode 选择
selected_by_task: dict[str, list[int]] = {}

for task_name in SPATIAL_TASK_NAMES:
    actual_task = available_by_normalized.get(
        normalize_task_name(task_name)
    )

    if actual_task is None:
        raise RuntimeError(
            f"Spatial task not found: {task_name}"
        )

    selected_by_task[actual_task] = choose_evenly_spaced(
        task_to_episodes[actual_task],
        TRAIN_EPISODES_PER_TASK,
    )

# 扁平的 episode index 汇总为排序后的列表
EPISODE_INDICES = sorted(
    episode_index
    for episode_indices in selected_by_task.values()
    for episode_index in episode_indices
)

# 预期: 10 tasks × 5 = 50 个
if len(EPISODE_INDICES) != 50:
    raise RuntimeError("Episode selection failed.")

print("Training data: 10 tasks × 5 episodes = 50 episodes")

## 8. 用 PyTorch 实现 SmolVLA

这是本教程的核心：参考论文和实现，用 PyTorch 逐步写出 **Vision-Language-Action 模型 SmolVLA**。

### 参考论文与实现

- **SmolVLA 架构** — Shukor et al., “SmolVLA: A Vision-Language-Action Model for Affordable and Efficient Robotics”（Hugging Face, 2025）— <https://huggingface.co/blog/smolvla>
- **Flow Matching（action head 生成模型）** — Lipman et al., “Flow Matching for Generative Modeling”（ICLR 2023）— <https://arxiv.org/abs/2210.02747>
- **Transformer self/cross-attention** — Vaswani et al., “Attention Is All You Need”（NeurIPS 2017）— <https://arxiv.org/abs/1706.03762>
- **LoRA（Low-Rank Adaptation）** — Hu et al., “LoRA: Low-Rank Adaptation of Large Language Models”（ICLR 2022）— <https://arxiv.org/abs/2106.09685>
- **LIBERO 基准** — Liu et al., “LIBERO: Benchmarking Knowledge Transfer for Lifelong Robot Learning”（NeurIPS 2023）— <https://arxiv.org/abs/2306.03310>
- **基准实现** — LeRobot v0.6.0 policies.smolvla — <https://github.com/huggingface/lerobot/tree/v0.6.0/src/lerobot/policies/smolvla>

### 整体结构

VLA 模型是两段流水线：

- **1. VLM（frozen）** — 输入：图像 $x \in \mathbb{R}^{H \times W \times 3}$ + 任务描述 $\tau$；输出：$h \in \mathbb{R}^{L \times 960}$。
  - 使用 SmolVLM2-500M 转成融合表示，参数不训练；
- **2. Action head（Flow Matching Head，可训练）** — 输入：noisy action chunk $x_t$、时间 $t$、条件 $(h,\ \text{state})$；输出：速度场 $v_\theta(x_t,t,h) \in \mathbb{R}^{K \times d_a}$。
  - 内部是多层 Transformer（self-attn + 对 VLM 的 cross-attn + MLP），再接输入/输出投影；
  - 通过 Flow Matching（Lipman et al. 2023）生成 action chunk。

其中 $K$ 是 `chunk_size`（未来 step 数），$d_a$ 是动作维度（LIBERO 为 7 = 6-DoF + gripper），$L$ 是 image patch + text token 数。

**要点**：VLM 冻结（不触碰 500M 参数），只训练 Action head 的数百万参数。

### Flow Matching 公式

对真实动作 $a \in \mathbb{R}^{K \times d_a}$ 和高斯噪声 $x_0 \sim \mathcal{N}(0,I)$，定义 $t \in [0,1]$ 的**直线路径**：

$$x_t=(1-t)x_0+t a$$

路径上的**真实速度**是：

$$v^*(x_t,t)=\frac{d x_t}{dt}=a-x_0$$

给定条件 $c$（图像、语言）后，让网络 $v_\theta(x_t,t,c)$ 回归 $v^*$，得到 Flow Matching Loss：

$$\mathcal{L}(\theta)=\mathbb{E}_{a\sim\mathcal{D},\ x_0\sim\mathcal{N},\ t\sim U(0,1)}\left[\left\|v_\theta(x_t,t,c)-(a-x_0)\right\|^2\right]$$

**推理**从纯噪声 $x_0 \sim \mathcal{N}(0,I)$ 出发，用 Euler 方法求解 ODE：

$$x_{t+\Delta t}=x_t+v_\theta(x_t,t,c)\cdot\Delta t$$

从 $t=0$ 推进到 $1$ 共 $N$ 步，就得到 $x_1\approx a$（通常 $N\in[5,20]$）。

### 为什么用 Flow Matching（而不是 Diffusion）

- **训练简单**：DDPM 需要处理每个时间的噪声 schedule 和方差，而 FM 只需学习“真实速度 = $a-x_0$”；
- **推理快速**：直线路径只需少量步骤（5–10）就容易得到不错的精度；
- **数值稳定**：不容易发散。

### 实现要点

- **冻结 VLM**：训练 500M 参数会同时增加资源消耗和过拟合风险；
- **只训练 Action head**：Transformer、输入/输出投影和位置/时间嵌入只包含数百万参数；
- **Action head 从零完整训练**：不使用 LoRA（详见 8.6）。

后续 cell 会逐个搭建这些组件。


### 8.1 时间嵌入

将标量时间 `t` 映射为高维的 sinusoidal 向量，让 Action head 知道当前位于 flow 的哪个阶段。这里的 TODO 只需要构造时间和频率的外积，shape 契约是 `(B, dim)`。


In [ ]:
# ==============================================================
# 8.1 时刻埋包含 (SinusoidalTimeEmbedding)
# ==============================================================
#
# 【目的】 連続时刻 t ∈ [0, 1] 网络到条条与并渡为了、高维度向量方向埋込。
#
# 【几式】  (D = hidden_dim, i = 0..D/2-1)
#     embed(t)[2i]   = sin(t / 10000^(2i/D))
#     embed(t)[2i+1] = cos(t / 10000^(2i/D))
#
# 【背景】
#   Transformer 的位置编码 (Vaswani et al. 2017) 「離散位置 → 連続时刻」到扩展完成也的。
#   Diffusion 模型中也常用作 timestep embedding (DDPM, Ho et al. 2020)。
#
# 【高维度到埋込还是】
#   标量 t 仅会导致输入層， `nn.Linear(1, D)` 等，扩展执行与会变成、
#   sin/cos 是「不同的周波几排列」与，、大局〜細部同时伝。训练也安定。

# section 8 全部部体，共通的 import
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SinusoidalTimeEmbedding(nn.Module):
    """标量时刻 t  hidden_dim 维度的向量到埋込."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        # sin 与 cos ，半分逐个埋的，偶几要求
        assert hidden_dim % 2 == 0, "hidden_dim 是偶几到并请"
        self.hidden_dim = hidden_dim  # 示示示示例: 960 (SmolVLM2 的 text hidden size)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # -------- 输入/输出的 shape 契约 --------
        # 输入 t:      shape = (B,)              batch 内的每个样个的时刻 (0〜1)
        # 输出:        shape = (B, hidden_dim)   每个样个的埋包含向量

        half = self.hidden_dim // 2  # 示示示示例: 480

        # ---- Step 1: 周波几的等比几列 ----
        # freqs[i] = exp(-log(10000) * i / half) = 1 / 10000^(i/half)
        # 高周波 (freqs[0]=1, 周期 2π) → 低周波 (freqs[half-1]≈1/10000, 周期 20000π)
        freqs = torch.exp(
            -math.log(10000.0)
            * torch.arange(half, device=t.device)  # shape: (half,)
            / half
        )
        # freqs shape: (half,)  示示示示例: (480,)

        # ---- Step 2: 时刻 × 周波几 的外積 ----
        # t[:, None]         shape: (B, 1)
        # freqs[None, :]     shape: (1, half)
        # 乗算 (broadcast)   shape: (B, half)
        # 【問題箇所 1】
        # 提示: broadcast 到 outer product
        raise NotImplementedError(
            "TODO 1: SinusoidalTimeEmbedding.forward ， args 計算"
        )
        args = ...    # ← 这里書換. shape: (B, half)

        # ---- Step 3: sin 与 cos 横到連結 ----
        # torch.sin(args)     shape: (B, half)
        # torch.cos(args)     shape: (B, half)
        # cat(dim=-1)         shape: (B, hidden_dim = 2*half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        # 最終 shape: (B, hidden_dim)  示示示示例: (B, 960)

### 8.2 Transformer block

Action token 先进行 self-attention，再通过 cross-attention 读取冻结 VLM 的视觉-语言 token，最后经过 MLP。保留 token 维度和 batch 维度的 shape 契约，是后续 action chunk 正常工作的关键。


In [ ]:
# ==============================================================
# 8.2 Transformer block
# ==============================================================
#
# 【1 層的中身】(Pre-LN 型 Transformer)
#     x -> LN -> self_attn -> +残差 -> LN -> cross_attn(VLM 参照) -> +残差
#       -> LN -> MLP        -> +残差 -> 输出
#
# 【 Q/K/V/O 分離执行还是】
#   nn.MultiheadAttention 是 in_proj_weight ， Q/K/V 合体并持为了、
#   attention 内的个按重触到 (可視化 / 診断 / 后述的 LoRA 追加実験 等面倒).
#   个按 nn.Linear 到并与重名称称前，参照，的，可読性上.
#
# 【Attention 的几式】(Vaswani et al. 2017)
#   Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V


class TransformerBlock(nn.Module):
    """Transformer decoder 1 層."""

    def __init__(self, hidden_dim: int, num_heads: int, mlp_ratio: int = 4):
        super().__init__()
        # multi-head 的分割，余出割切需要有
        assert hidden_dim % num_heads == 0, "hidden_dim 是 num_heads 的倍几到并请"

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        # 1 head 扱维度 (示示示示例: hidden=960, heads=8 → head_dim=120)
        self.head_dim = hidden_dim // num_heads

        # ---- Self-attention 的 4 的 Linear ----
        # bias=False 是 LLaMA/SmolVLM 系列列的慣示示示示例 (参几节省 & 训练安定)
        self.self_q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # (D → D)
        self.self_k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # (D → D)
        self.self_v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # (D → D)
        self.self_o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # (D → D)

        # ---- Cross-attention 的 4 的 Linear ----
        # Q 是 action tokens 从、K/V 是 VLM hidden state 从作
        # (「画像+言語条条 action 到注意意入执行」gate)
        self.cross_q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.cross_k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.cross_v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.cross_o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # ---- MLP (position-wise feed-forward, 4倍→GELU→恢复) ----
        # D → 4D → D
        self.mlp_in  = nn.Linear(hidden_dim, mlp_ratio * hidden_dim, bias=False)
        self.mlp_out = nn.Linear(mlp_ratio * hidden_dim, hidden_dim, bias=False)

        # ---- LayerNorm (Pre-LN 型) ----
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # 输入  shape: (B, L, D)              -- D = num_heads * head_dim
        # 输出  shape: (B, num_heads, L, head_dim)
        #
        # multi-head attention 是 head 軸独立並列計算执行为了、
        # 通道軸 (head, head_dim) 到分割并並替。
        B, L, D = x.shape
        return x.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        # after view:      (B, L, num_heads, head_dim)
        # after transpose: (B, num_heads, L, head_dim)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        # 输入  shape: (B, num_heads, L, head_dim)
        # 输出  shape: (B, L, D)               -- D = num_heads * head_dim
        B, H, L, Dh = x.shape
        return x.transpose(1, 2).contiguous().view(B, L, H * Dh)

    def _attention(
        self,
        q_proj: nn.Linear, k_proj: nn.Linear, v_proj: nn.Linear, o_proj: nn.Linear,
        q_input: torch.Tensor,   # (B, Lq, D)  ← Q 的元
        k_input: torch.Tensor,   # (B, Lkv, D) ← K 的元
        v_input: torch.Tensor,   # (B, Lkv, D) ← V 的元
    ) -> torch.Tensor:           # (B, Lq, D)
        # Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V 計算.
        # 个々的射影与 head 分割是自前，行、softmax 仅 SDPA 到任.

        # ---- Step 1: 射影 → head 分割 ----
        # 射影后:     (B, L, D)
        # 分割后:     (B, H, L, D_h)  (H = num_heads, D_h = head_dim)
        Q = self._split_heads(q_proj(q_input))   # (B, H, Lq,  D_h)
        K = self._split_heads(k_proj(k_input))   # (B, H, Lkv, D_h)
        V = self._split_heads(v_proj(v_input))   # (B, H, Lkv, D_h)

        # ---- Step 2: Scaled Dot-Product Attention ----
        # 几式: softmax(Q K^T / sqrt(D_h)) V
        # SDPA 是几值安定 + Flash Attention v2 自動采用执行 PyTorch 标准実装
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        # attn_out shape: (B, H, Lq, D_h)

        # ---- Step 3: head 統合 → 输出射影 ----
        merged = self._merge_heads(attn_out)     # (B, Lq, D)
        return o_proj(merged)                    # (B, Lq, D)

    def forward(
        self,
        action_tokens: torch.Tensor,  # (B, chunk_size, D)   ← chunk 内的 action 表現
        vlm_hidden:    torch.Tensor,  # (B, T_vlm,     D)   ← VLM 输出 (画像+text 融合)
    ) -> torch.Tensor:                # (B, chunk_size, D)
        # 以下全部部部写成 Pre-LN 结构 (norm → sub-layer → residual)。
        # 残差连接到層深并也勾配消失到 (Deep Residual Learning, He et al. 2016).

        # ---- Sub-layer 1: Self-attention (chunk 内的相互作用) ----
        # action tokens 「自分个身」 Q/K/V 到并相互参照执行
        h = self.norm1(action_tokens)                                    # (B, chunk, D)
        action_tokens = action_tokens + self._attention(
            self.self_q_proj, self.self_k_proj,
            self.self_v_proj, self.self_o_proj,
            h, h, h,
        )                                                                 # (B, chunk, D)

        # ---- Sub-layer 2: Cross-attention (VLM 参照 = 条条付) ----
        # 这里，画像+言語的情報吸. Q 是 action, K/V 是 VLM.
        h = self.norm2(action_tokens)                                    # (B, chunk, D)
        action_tokens = action_tokens + self._attention(
            self.cross_q_proj, self.cross_k_proj,
            self.cross_v_proj, self.cross_o_proj,
            h,           # (B, chunk, D)  ← Q
            vlm_hidden,  # (B, T_vlm, D)  ← K
            vlm_hidden,  # (B, T_vlm, D)  ← V
        )                                                                 # (B, chunk, D)

        # ---- Sub-layer 3: Position-wise MLP ----
        # 每个位置到独立到非線形转换还是. (D → 4D → GELU → D)
        # GELU (Hendrycks & Gimpel 2016) 是 Transformer 系列列，的定番活性化関几
        h = self.norm3(action_tokens)                                    # (B, chunk, D)
        mlp_h = F.gelu(self.mlp_in(h))                                   # (B, chunk, 4D)
        action_tokens = action_tokens + self.mlp_out(mlp_h)              # (B, chunk, D)

        return action_tokens                                             # (B, chunk, D)

### 8.3 Flow Matching Head

这个 head 输入 noisy action chunk、时间嵌入和 VLM 条件，预测每个 action token 的速度。训练时回归 `a - x0`，推理时从高斯噪声开始，用多次 Euler 更新生成动作序列。


In [ ]:
# ==============================================================
# 8.3 Flow Matching Head
# ==============================================================
#
# 【役割】 noise → action 到「速度場 v_theta(x_t, t, cond)」予測执行 Transformer.
#
# 【整体 shape 转换】
#     输入:     (B, K, d_a)              -- noisy actions
#     射影后:   (B, K, D)                -- action_in ， d_a→D
#     +pos_emb: (B, K, D)                -- chunk 内位置埋包含
#     +time_emb:(B, K, D)                -- 时刻每个 chunk 位置到 broadcast
#     ↓ (Transformer Block  num_layers 段)
#     (B, K, D)                          -- VLM 参照精緻化
#     ↓ action_out ， D→d_a
#     输出:     (B, K, d_a)              -- velocity


class FlowMatchingHead(nn.Module):
    """action chunk 的 velocity 場条条带有予測执行 Transformer."""

    def __init__(
        self,
        hidden_dim: int,     # D. VLM 的 hidden 维度与揃 (SmolVLM2 若 960)
        action_dim: int,     # d_a. LIBERO 是 7 (6-DoF + gripper)
        chunk_size: int,     # K. 未来何 step 部分的 action 一気到予測执行还是
        num_layers: int,     # Transformer 的層几 (増和与表現力↑ / 計算量↑)
        num_heads: int,      # multi-head 的 head 几 (D 割切需要)
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.action_dim = action_dim
        self.chunk_size = chunk_size

        # ---- action 维度 (d_a=7) ⇄ hidden 维度 (D=960) 的入输出射影 ----
        # 入输出的解釈维度橋渡执行部分. head 个体与是役割不同的，独立到扱.
        self.action_in  = nn.Linear(action_dim, hidden_dim)  # (7 → 960)
        self.action_out = nn.Linear(hidden_dim, action_dim)  # (960 → 7)

        # ---- chunk 内的位置埋包含 (learnable parameter, zero-init) ----
        # Transformer 是 permutation invariant 因此位置情報明示的到注意意入执行需要有.
        # 実 SmolVLA ，是 RoPE (Rotary Position Embedding, Su et al. 2021) 使、
        # 教材，是简单到 learnable absolute embedding.
        # shape: (1, chunk_size, hidden_dim)  -- batch 軸是 broadcast ，支持
        self.pos_embed = nn.Parameter(torch.zeros(1, chunk_size, hidden_dim))

        # ---- 时刻 t 用的埋包含器 (section 8.1) ----
        self.time_embed = SinusoidalTimeEmbedding(hidden_dim)

        # ---- Transformer 个体 (num_layers 段) ----
        # 每个層的重是 blocks[i].self_q_proj 等个按 Linear ，参照可能.
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(
        self,
        noisy_actions: torch.Tensor,  # (B, K,     d_a)   -- 中間噪声状態 x_t
        t:             torch.Tensor,  # (B,)              -- 时刻 t ∈ [0, 1]
        vlm_hidden:    torch.Tensor,  # (B, T_vlm, D)     -- VLM 输出 (条条)
    ) -> torch.Tensor:                # (B, K,     d_a)   -- velocity 予測
        # ---- Step 1: action 维度 → hidden 投影到二维 ----
        # action_in(noisy_actions):  (B, K, d_a) → (B, K, D)
        x = self.action_in(noisy_actions) + self.pos_embed
        # x shape: (B, K, D)
        # pos_embed shape: (1, K, D)  batch 軸到 broadcast 被加算被

        # ---- Step 2: 时刻埋包含 chunk 全部部 step 到broadcast加算 ----
        # time_embed(t):        (B, D)
        # time_embed(t)[:, None, :]:  (B, 1, D)  ← chunk 軸到 1 挟
        # x 到 broadcast 加算   → (B, K, D)
        x = x + self.time_embed(t)[:, None, :]

        # ---- Step 3: Transformer 層， VLM 情報吸 ----
        # 每个層， self-attn + cross-attn(VLM 参照) + MLP 通.
        # shape 是每个層内，保存被: (B, K, D)
        for block in self.blocks:
            x = block(x, vlm_hidden)  # (B, K, D)

        # ---- Step 4: hidden 维度 → action 维度方向戻并 velocity 与并输出 ----
        # action_out:  (B, K, D) → (B, K, d_a)
        return self.action_out(x)

### 8.4 加载并完全冻结 SmolVLM2

加载 `SmolVLM2-500M-Video-Instruct` 作为视觉-语言 backbone。它只提供条件表示，不参与反向传播；这样训练资源集中在本课题自己实现的 Action head 上。


In [ ]:
# ==============================================================
# 8.4 SmolVLM2-500M-Video-Instruct - + 完全部部 freeze
# ==============================================================
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image
import numpy as np

VLM_MODEL_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

# bf16 支持 GPU (A100/H100/L40/RTX40 系列列) 若 bf16、否则 fp16
# fp32 会导致 500M × 4byte = 2GB 与内存食的，現実的，
_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading VLM ({VLM_MODEL_ID}) in {_dtype}...", flush=True)

# processor: 画像调整尺寸・归一化・token化全部部部担当执行統合对象
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

# 个体
vlm_model = AutoModelForImageTextToText.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=_dtype,
).to("cuda")

# ---- 完全部部 freeze (个课题的縛: VLM 是训练对象外) ----
vlm_model.eval()
for p in vlm_model.parameters():
    p.requires_grad = False

# hidden_dim = 960 (以降的 action head 的入输出维度)
VLM_HIDDEN_DIM = vlm_model.config.text_config.hidden_size
print(f"VLM loaded. hidden_dim = {VLM_HIDDEN_DIM}")

# 训练可能 params  0 个，有与确认
_train = sum(p.numel() for p in vlm_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in vlm_model.parameters())
print(f"VLM params: total {_total/1e6:.1f}M, trainable {_train/1e6:.1f}M  ← 0 的是")


class RealVLMEncoder(nn.Module):
    """SmolVLM2 封装: (画像 list, task 文 list) → hidden state.

    - VLM 是冻结済因此 forward 是 torch.no_grad
    - 内部， processor  chat template 組立、pixel_values / input_ids 到转换
    - 最后的 layer 的 hidden_states (画像 tokens + text tokens 的融合表現) 返次
    """

    def __init__(self, model, processor, hidden_dim: int, dtype):
        super().__init__()
        self.model = model
        self.processor = processor
        self.hidden_dim = hidden_dim
        self._dtype = dtype

    @torch.no_grad()
    def forward(self, images, task_texts: list[str]) -> torch.Tensor:
        # images: PIL / numpy / torch tensor 也受付 (batch是 list ，渡)
        # task_texts: 每个样个的任务説明文 (長 B 的 list)

        # 输入 PIL 到統一 (processor 是 PIL 预期执行)
        pil_images = []
        for img in images:
            if isinstance(img, np.ndarray):
                pil_images.append(Image.fromarray(img.astype(np.uint8)))
            elif isinstance(img, torch.Tensor):
                arr = img.cpu().numpy()
                if arr.ndim == 3 and arr.shape[0] == 3:  # (C,H,W) → (H,W,C)
                    arr = arr.transpose(1, 2, 0)
                pil_images.append(Image.fromarray(arr.astype(np.uint8)))
            else:
                pil_images.append(img)

        # SmolVLM2 的 chat template ，「画像 + 指令」的 prompt 到組立
        prompts = []
        for txt in task_texts:
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": txt},
                ],
            }]
            prompts.append(
                self.processor.apply_chat_template(
                    messages, add_generation_prompt=True
                )
            )

        # tokenize + preprocess. padding=True ，可変長揃
        inputs = self.processor(
            text=prompts,
            images=pil_images,
            return_tensors="pt",
            padding=True,
        )

        # 模型的 device / dtype 到揃
        device = next(self.model.parameters()).device
        inputs = {
            k: (v.to(device=device, dtype=self._dtype)
                if v.dtype in (torch.float16, torch.float32, torch.bfloat16)
                else v.to(device))
            for k, v in inputs.items()
        }

        # 最后的 hidden state 取的， output_hidden_states=True
        outputs = self.model(**inputs, output_hidden_states=True)

        # hidden_states 是每个 layer 分的 tuple、[-1] 最終 layer
        # shape: (B, L, hidden_dim). L 是画像 tokens + 文个 tokens 的合计長.
        return outputs.hidden_states[-1]


# 動作确认: 適当画像与任务文，代码
vlm_encoder = RealVLMEncoder(vlm_model, vlm_processor, VLM_HIDDEN_DIM, _dtype)

_dummy_img = [Image.new("RGB", (256, 256), color=(128, 64, 200))]
_dummy_task = ["pick up the black bowl on the plate"]
_hidden = vlm_encoder(_dummy_img, _dummy_task)
print(f"VLM output shape: {tuple(_hidden.shape)}  dtype: {_hidden.dtype}")
# 预期: (1, L, 960). L 是prompt長 + 画像token几的合计

### 8.5 组装 VLA

把冻结的 VLM、状态投影和 Flow Matching Action head 连接起来。重点检查：图像/语言条件是否与 batch 对齐，action chunk 是否为 `(B, K, d_a)`，以及训练和 rollout 是否使用同一个归一化/反归一化契约。

四个 TODO 分别对应 Flow Matching 的核心公式：中间点、真实速度和 Euler 更新，以及时间嵌入所需的外积。不要改变填空范围或训练参数。


In [ ]:
# ==============================================================
# 8.5 SmolVLA 統合模块
# ==============================================================
#
# 【役割】 Vision-Language (frozen SmolVLM2) + State + Action (自作 FlowMatchingHead) 結合.
#
# 【VLA 的 "S"】 SmolVLA 是画像・言語到加 **当前机器人状態 (state)** 也输入到取.
#   典型的中只有 joint positions (7) + gripper qpos (1) = 8 维度. LIBERO 几据集的
#   observation.state 与一致执行. state  Linear ， transformer_hidden_dim 到射影、
#   VLM 输出的末尾到 1 的 state token 与并連結执行.
#   (概念的中只有 SmolVLA 官方文档的与相同発想、官方文档的，是 VLM 内部処理与統合被
#    点不同的. 詳細是 section 8.5 的对照表参照.)
#
# 【训练用 compute_loss 的几式】
#     L(theta) = || v_theta(x_t, t, cond) - (a - x_0) ||^2
#     where  x_0 ~ N(0, I),  t ~ U(0, 1),  x_t = (1-t) x_0 + t a,
#            cond = [vlm_projector(VLM(images, task_text)),  state_proj(state)]
#
# 【推理用 sample_actions 的几式】(Euler 積分, t: 0 -> 1)
#     x <- sample from N(0, I)                          [初期化]
#     for step in 0..N-1:
#         t = step / N
#         x <- x + v_theta(x, t, cond) * (1/N)          [Euler 1 歩]
#     return x   # ≈ 真的 action


class SmolVLA(nn.Module):
    """SmolVLA 全部部体. VLM 是 frozen、Action head 是 trainable、state 是 projection 経由."""

    def __init__(
        self,
        vlm_encoder: "RealVLMEncoder",   # 冻结済wrapper (section 8.4)
        action_dim: int = 7,             # LIBERO: 6-DoF + gripper = 7 维度
        state_dim: int = 8,              # LIBERO 的 observation.state (joint 7 + gripper 1)
        chunk_size: int = 8,             # 予測执行連続 action 几 (実 SmolVLA 是 50)
        num_transformer_layers: int = 4, # Transformer 的層几
        num_heads: int = 8,              # multi-head 的 head 几
        transformer_hidden_dim: int | None = None,  # None 若 VLM 的 hidden dim (960) 该使
    ):
        super().__init__()
        self.vlm_encoder = vlm_encoder

        # VLM 的元 hidden (SmolVLM2 若 960)
        vlm_hidden = vlm_encoder.hidden_dim

        # Transformer 内部的 hidden 独立到指定，
        if transformer_hidden_dim is None:
            transformer_hidden_dim = vlm_hidden
        self.transformer_hidden_dim = transformer_hidden_dim

        # VLM 输出 (dim=960)  Transformer 用的维度 (示示示示例 256) 到線形射影
        if transformer_hidden_dim != vlm_hidden:
            self.vlm_projector = nn.Linear(vlm_hidden, transformer_hidden_dim, bias=False)
        else:
            self.vlm_projector = nn.Identity()

        # ★ state  transformer_hidden_dim 到射影执行層 (VLA 的 "S" 部分)
        # 8 维度的 robot state → 256 维度的埋包含token 1 个
        self.state_proj = nn.Linear(state_dim, transformer_hidden_dim)

        # 后方互換的为了 hidden_dim 也 expose
        self.hidden_dim = transformer_hidden_dim
        self.action_dim = action_dim
        self.state_dim = state_dim
        self.chunk_size = chunk_size

        # Flow Matching Head 是 Transformer 的 hidden 维度，構築
        self.flow_head = FlowMatchingHead(
            hidden_dim=transformer_hidden_dim,
            action_dim=action_dim,
            chunk_size=chunk_size,
            num_layers=num_transformer_layers,
            num_heads=num_heads,
        )

    def _encode_cond(
        self, images, task_texts, states: torch.Tensor
    ) -> torch.Tensor:
        # 1) VLM 从画像+言語的融合表現 (勾配无)
        hidden = self.vlm_encoder(images, task_texts)   # (B, L, 960)
        # 2) dtype  action head 到合 (VLM: fp16/bf16, head: fp32)
        target_dtype = next(self.flow_head.parameters()).dtype
        hidden = hidden.to(dtype=target_dtype)
        # 3) Transformer 的 hidden 维度到射影 (Identity or Linear)
        cond_vlm = self.vlm_projector(hidden)            # (B, L, transformer_hidden_dim)
        # 4) state 射影并 1 token与并末尾到連結
        state_token = self.state_proj(
            states.to(dtype=cond_vlm.dtype, device=cond_vlm.device)
        )                                                # (B, transformer_hidden_dim)
        state_token = state_token.unsqueeze(1)           # (B, 1, transformer_hidden_dim)
        # 5) [VLM token列 | state token 1 个]  connect
        return torch.cat([cond_vlm, state_token], dim=1)  # (B, L+1, transformer_hidden_dim)

    def compute_loss(
        self,
        images,                         # 長 B 的 list (PIL/np/tensor 何也可)
        task_texts: list[str],          # 長 B 的 list (每个样个的 task 説明文)
        states: torch.Tensor,           # (B, state_dim) 当前机器人状態
        actions: torch.Tensor,          # (B, K, d_a)  归一化済真 action
    ) -> torch.Tensor:                  # scalar (MSE loss)
        """Flow Matching 训练損失.

        算法 (Lipman et al. 2023, Conditional Flow Matching):
          1. 每个样个到噪声 x_0 ~ N(0, I) 用意                     (B, K, d_a)
          2. 每个样个到时刻 t ~ Uniform(0, 1) 用意                   (B,)
          3. 線形路径上的中間点 x_t = (1-t) x_0 + t a 計算            (B, K, d_a)
          4. 真的速度 v* = a - x_0 (直線路径因此 path 上，定几)      (B, K, d_a)
          5. VLM+state 的融合条条 cond 获取                            (B, L+1, D)
          6. 网络到 (x_t, t, cond) 渡并速度予測 v_pred    (B, K, d_a)
          7. MSE(v_pred, v*) 返次                                      scalar
        """
        B = actions.size(0)
        device = actions.device

        # ---- 条条 (VLM + state, 画像/言語/状態的融合) ----
        vlm_hidden = self._encode_cond(images, task_texts, states).to(device)
        # vlm_hidden shape: (B, L+1, transformer_hidden_dim)

        # ---- Step 1: 噪声与时刻样个 ----
        noise = torch.randn_like(actions)     # (B, K, d_a)  x_0 ~ N(0, I)
        t = torch.rand(B, device=device)      # (B,)         t ~ U(0, 1)

        # ---- Step 2: 線形補間，中間点 x_t 创建 ----
        tv = t[:, None, None]                 # (B, 1, 1) — chunk / action 軸到 broadcast
        # 【問題箇所 2】
        # 几式: x_t = (1 - t) * x_0 + t * a       (tv=t, noise=x_0, actions=a)
        # 输出 shape: (B, K, d_a)
        raise NotImplementedError(
            "TODO 2: 線形補間 x_t 実装 (SmolVLA.compute_loss)"
        )
        xt = ...    # ← 这里書換

        # ---- Step 3: 真的速度 = a - x_0 (直線路径因此定几場) ----
        # 【問題箇所 3】
        # 几式: v* = a - x_0     (path 上，定几)
        # 输出 shape: (B, K, d_a)
        raise NotImplementedError(
            "TODO 3: 真的速度 target_v 計算 (SmolVLA.compute_loss)"
        )
        target_v = ...    # ← 这里書換

        # ---- Step 4: 网络，速度予測 ----
        pred_v = self.flow_head(xt, t, vlm_hidden)  # (B, K, d_a)

        # ---- Step 5: MSE loss ----
        return F.mse_loss(pred_v, target_v)

    @torch.no_grad()
    def sample_actions(
        self,
        images,
        task_texts: list[str],
        states: torch.Tensor,             # (B, state_dim) 当前机器人状態
        num_steps: int = 10,
    ) -> torch.Tensor:                 # (B, K, d_a)  归一化空间的 action chunk
        """Euler 法， ODE 几值積分并 action chunk 恢复."""
        device = next(self.flow_head.parameters()).device

        # VLM+state 条条 (勾配无, 预计算并使旋转)
        vlm_hidden = self._encode_cond(images, task_texts, states).to(device)
        # vlm_hidden shape: (B, L+1, transformer_hidden_dim)

        B = vlm_hidden.size(0)

        # ---- 初期化: pure noise ----
        x = torch.randn(B, self.chunk_size, self.action_dim, device=device)
        # x shape: (B, K, d_a)

        dt = 1.0 / num_steps  # 示示示示例: N=10 若 dt = 0.1

        # ---- Euler 積分- ----
        for step in range(num_steps):
            t_now = torch.full((B,), step * dt, device=device)  # (B,)
            v = self.flow_head(x, t_now, vlm_hidden)             # (B, K, d_a)
            # 【問題箇所 4】
            # 几式: x_{t+dt} = x_t + v(x, t, cond) * dt
            # 输出 shape: (B, K, d_a)
            raise NotImplementedError(
                "TODO 4: Euler 的 1 歩実装 (SmolVLA.sample_actions)"
            )
            x = ...    # ← 这里書換

        # 最終 x 是「正規已变为 action 空间」的值.
        # 呼出側， denormalize_actions() 并実 action 到恢复.
        return x


# ==============================================================
# 实示示示例化: 論文大小 (注意释掉) 与 教材大小 対比
# ==============================================================

# ---- 論文相当大小 (SmolVLA / LeRobot 官方文档的默认、参考) ----
#
# smolvla = SmolVLA(
#     vlm_encoder=vlm_encoder,
#     action_dim=7,
#     state_dim=8,
#     chunk_size=50,                # SmolVLA original: 50 step (2.5 秒 @ 20Hz)
#     num_transformer_layers=16,    # SmolVLA original: num_vlm_layers 与相同
#     num_heads=15,                 # SmolVLA original: hidden 720 / head_dim 48
#     transformer_hidden_dim=720,   # VLM_hidden × 0.75 (官方文档的 expert_width_multiplier=0.75)
# ).to("cuda")

# ---- 教材用大小 (from-scratch × 50 episode 向到小型化) ----
smolvla = SmolVLA(
    vlm_encoder=vlm_encoder,
    action_dim=7,
    state_dim=8,                    # LIBERO 的 observation.state 维度
    chunk_size=8,                   # 8 step 予測 (0.4 秒). 训练也推理也軽
    num_transformer_layers=4,       # 16 → 4 到圧縮
    num_heads=8,                    # transformer_hidden_dim=256 到合 (256/32 = 8)
    transformer_hidden_dim=256,     # 720 → 256 到圧縮 (params 几分的 1 到)
).to("cuda")

# ---- 训练可能 params 与冻结 params 的内分 ----
_train = sum(p.numel() for p in smolvla.parameters() if p.requires_grad)
_frozen = sum(p.numel() for p in smolvla.parameters() if not p.requires_grad)
print(f"SmolVLA — trainable: {_train/1e6:.2f}M  frozen: {_frozen/1e6:.2f}M")
print(f"  (frozen  SmolVLM2-500M 部分、trainable  Action head + projector + state_proj)")
print(f"  transformer_hidden_dim = {smolvla.transformer_hidden_dim}, "
      f"chunk_size = {smolvla.chunk_size}, "
      f"state_dim = {smolvla.state_dim}, "
      f"num_transformer_layers = {len(smolvla.flow_head.blocks)}")

### 8.6 检查可训练参数

确认 VLM 参数全部 frozen，只有 Action head、投影层和嵌入层被列为 trainable。若冻结状态不符合预期，先在继续训练前检查模型组装。


In [ ]:
# ==============================================================
# 8.6 训练对象参几的内分确认执行
# ==============================================================
# 预期被内分:
#   frozen    ≈ 500 M   ← SmolVLM2-500M (视觉+语言部分，全部部部冻结)
#   trainable ≈ 几 M    ← FlowMatchingHead (自定义 action head，全部部部训练)


def summarize_params(model: nn.Module, top_k: int = 8) -> None:
    """训练可能 / 冻结参几的合计与、top K 模块的内分显示."""
    total_trainable = 0
    total_frozen = 0
    per_module_trainable: dict[str, int] = {}

    for name, p in model.named_parameters():
        n = p.numel()
        if p.requires_grad:
            total_trainable += n
            # 模块単位 (先頭 2 階層) ，集約
            key = ".".join(name.split(".")[:2])
            per_module_trainable[key] = per_module_trainable.get(key, 0) + n
        else:
            total_frozen += n

    total = total_trainable + total_frozen
    pct = 100.0 * total_trainable / max(total, 1)
    print(f"Total params    : {total/1e6:8.2f} M")
    print(f"  Frozen (VLM)  : {total_frozen/1e6:8.2f} M")
    print(f"  Trainable     : {total_trainable/1e6:8.2f} M  ({pct:.2f}% of total)")
    print()

    # 训练对象模块大順到 top_k 个显示
    print(f"Trainable module breakdown (top {top_k}):")
    ranked = sorted(per_module_trainable.items(), key=lambda kv: -kv[1])
    for name, n in ranked[:top_k]:
        print(f"  {name:40s} {n/1e6:8.3f} M")


summarize_params(smolvla)

# 全部部训练参几 flow_head 配下到有与确认 (VLM 側是 0)
_vlm_train = sum(
    p.numel() for p in smolvla.vlm_encoder.parameters() if p.requires_grad
)
_head_train = sum(
    p.numel() for p in smolvla.flow_head.parameters() if p.requires_grad
)
print()
print(f"vlm_encoder trainable: {_vlm_train/1e6:.3f} M  ← 0 的是")
print(f"flow_head   trainable: {_head_train/1e6:.3f} M  ← 这里训练被")

## 9. 训练

使用前面构建的模型和 LIBERO 数据进行少量训练。先准备 action chunk 与归一化统计，再运行 PyTorch 训练循环。


### 9.1 加载 LIBERO 数据集

用 `LeRobotDataset` 提供带有 `chunk_size` 个 future action 的样本。训练与 rollout 必须保持相同的观测键、shape 和 action 顺序。


In [ ]:
# ==============================================================
# 9.1 LIBERO 几据集 (chunk_size 部分的 future action 付)
# ==============================================================
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# smolvla 的 chunk_size 与揃 (揃与 loss 計算， shape mismatch)
CHUNK_SIZE_TRAIN = smolvla.chunk_size  # 8

# LIBERO 的記録频率 = 20 Hz. delta_timestamps 是「秒単位的偏移」，指定执行的，
# i-th 未来 step 是 i / fps 秒后.
_DATASET_FPS = 20.0
delta_timestamps = {
    "action": [i / _DATASET_FPS for i in range(CHUNK_SIZE_TRAIN)],
}

print(f"Loading LIBERO dataset with {len(EPISODE_INDICES)} episodes...", flush=True)

# dataset 首次加载会从 HF 拉取对应 episode 的视频，因此需要一些时间。
# 该进度見为了、的-仅 progress bar 启用.
from huggingface_hub.utils import enable_progress_bars, disable_progress_bars
enable_progress_bars()
try:
    # section 7 ，选择完成 50 episode仅使
    train_dataset = LeRobotDataset(
        DATASET_REPO,
        revision=DATASET_REVISION,
        episodes=EPISODE_INDICES,
        delta_timestamps=delta_timestamps,
    )
finally:
    disable_progress_bars()

# 1 样个的中身确认
_sample = train_dataset[0]
print("\n=== sample keys ===")
for k, v in _sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:40s} shape={tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:40s} type={type(v).__name__}  val={str(v)[:60]}")

print(f"\n# training samples total: {len(train_dataset)}")

# --- state 定義的可視化 (rollout 時与一致执行还是要确认) ---
_state0 = _sample["observation.state"]
print(f"\n=== observation.state 的実样个 ===")
print(f"  shape:  {tuple(_state0.shape)}")
print(f"  values: {_state0.numpy().round(3)}")
print("↑ 个教材，是 LIBERO env 的 joints.pos (7) + gripper.qpos[0] (1) ")
print("   相同順序， 8 维度 state 与并構成执行 (section 10.1 参照).")

### 9.2 计算 action 的 mean / std（归一化）

从训练数据计算 action 的均值和标准差。模型在归一化空间学习，推理结束后必须使用同一组统计量反归一化。


In [ ]:
# ==============================================================
# 9.2 action 的平均・分散，归一化执行辅助函几创建
# ==============================================================
# 50 episode全部部部的 action 集 (chunk_size × N, action_dim) 到 flatten 、
# 维度每个 mean / std 計算执行.

print("Collecting action statistics...", flush=True)
_all_actions = []
for i in range(len(train_dataset)):
    _all_actions.append(train_dataset[i]["action"])
_all_actions = torch.stack(_all_actions).view(-1, 7)   # (N, 7)

# std  0 会导致割算， NaN 出现的，下限設
action_mean = _all_actions.mean(dim=0)                 # (7,)
action_std = _all_actions.std(dim=0).clamp_min(1e-3)   # (7,)

# GPU 上到置 (训练期间的毎 step 的归一化到使为了)
action_mean = action_mean.to("cuda", dtype=torch.float32)
action_std = action_std.to("cuda", dtype=torch.float32)

print(f"action_mean: {action_mean.cpu().numpy().round(3)}")
print(f"action_std : {action_std.cpu().numpy().round(3)}")


def normalize_actions(a: torch.Tensor) -> torch.Tensor:
    # 训练前：用 std 除以真 action - mean，结果应接近均值 0、方差 1。
    return (a - action_mean) / action_std


def denormalize_actions(a: torch.Tensor) -> torch.Tensor:
    # 推理后: 模型输出到 std 掛 mean 足与元尺度方向
    return a * action_std + action_mean

### 9.3 PyTorch 训练循环

运行 AdamW、warmup+cosine 学习率与梯度裁剪。基础版的目标是验证实现和日志契约能够走通，不要把少量训练的成功率当作最终性能结论。


In [ ]:
# ==============================================================
# 9.3 PyTorch 训练-
# ==============================================================
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import time

# ---- PyTorch 版个个的训练超参几 ----
# 这里，定義执行 TRAIN_STEPS_PY 等 PyTorch 自作版个个 (的cell) 制御执行.
TRAIN_STEPS_PY = 300           # 训练步几几 (示范用到控. 个気若 3000)
TRAIN_BATCH_SIZE_PY = 1        # VLM 很大的， batch=1 無難
TRAIN_LR_PY = 3e-4             # peak lr
TRAIN_WARMUP_PY = 30           # linear warmup 区間
TRAIN_LOG_FREQ_PY = 20         # loss  print 执行频率
TRAIN_NUM_WORKERS_PY = 0       # Colab 环境，是 0 安全部部

PY_CHECKPOINT_DIR = WORKDIR / "py_train"
PY_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ---- DataLoader ----
# LeRobotDataset 是 torch Dataset 与并使的， DataLoader 普通到組
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE_PY,
    shuffle=True,
    num_workers=TRAIN_NUM_WORKERS_PY,
    drop_last=True,
)

# ---- optimizer: 训练可能 params 仅渡 ----
# 这里，是 FlowMatchingHead 个体 + vlm_projector + state_proj + pos_embed 对象.
# VLM 是 freeze 并有的，勝手到 requires_grad=False ，除外被.
_trainable_params = [p for p in smolvla.parameters() if p.requires_grad]
print(f"Optimizer target: {sum(p.numel() for p in _trainable_params)/1e6:.2f}M params")

optimizer = AdamW(_trainable_params, lr=TRAIN_LR_PY, weight_decay=1e-4)


# ---- LR schedule: linear warmup → cosine decay ----
def _lr_lambda(step: int):
    if step < TRAIN_WARMUP_PY:
        # warmup 期间 0 → 1 到線形上昇
        return step / max(TRAIN_WARMUP_PY, 1)
    # warmup 后是 cosine ， 1.0 → 0.1 (peak lr 的 10% 到下)
    progress = (step - TRAIN_WARMUP_PY) / max(TRAIN_STEPS_PY - TRAIN_WARMUP_PY, 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return 0.1 + 0.9 * cosine


scheduler = LambdaLR(optimizer, lr_lambda=_lr_lambda)


# ---- 训练- ----
smolvla.train()
_step = 0
_loss_history = []
_start = time.time()

print(f"\nTraining {TRAIN_STEPS_PY} steps (batch={TRAIN_BATCH_SIZE_PY})...", flush=True)

while _step < TRAIN_STEPS_PY:
    for batch in train_loader:
        if _step >= TRAIN_STEPS_PY:
            break

        # 画像 PIL/numpy 到整形 (RealVLMEncoder 受付形式)
        # LeRobotDataset 是 (B, C, H, W) float in [0, 1] 返次
        # LIBERO dataset 的相机-是 "front" (俯瞰) 与 "wrist" (末端).
        # 教材版个个，是 front 仅使 (multi-view 是改造主题的一)
        imgs_tensor = batch["observation.images.front"]
        if imgs_tensor.dtype.is_floating_point:
            imgs_tensor = (imgs_tensor * 255).clamp(0, 255).to(torch.uint8)
        imgs_np = imgs_tensor.permute(0, 2, 3, 1).cpu().numpy()  # (B, H, W, C)
        imgs_list = [imgs_np[i] for i in range(imgs_np.shape[0])]

        # 任务説明文
        tasks_list = batch["task"]
        if isinstance(tasks_list, str):
            tasks_list = [tasks_list]

        # action  GPU 到 + 归一化
        actions = batch["action"].to("cuda", dtype=torch.float32)
        actions_norm = normalize_actions(actions)

        # ★ state (当前机器人状態)  GPU 到. LIBERO 是 (B, 8).
        states = batch["observation.state"].to("cuda", dtype=torch.float32)

        # forward → loss → backward
        loss = smolvla.compute_loss(imgs_list, tasks_list, states, actions_norm)

        optimizer.zero_grad()
        loss.backward()
        # 勾配爆発対策 (max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(_trainable_params, max_norm=1.0)
        optimizer.step()
        scheduler.step()

        _loss_history.append(loss.item())

        # 日志输出: LOG_FREQ step 每个 loss / lr / elapsed  print
        if (_step + 1) % TRAIN_LOG_FREQ_PY == 0 or _step == 0:
            _lr = scheduler.get_last_lr()[0]
            _elapsed = time.time() - _start
            _avg_loss = sum(_loss_history[-TRAIN_LOG_FREQ_PY:]) / min(
                len(_loss_history), TRAIN_LOG_FREQ_PY
            )
            print(
                f"step {_step+1:4d}/{TRAIN_STEPS_PY}  "
                f"loss={_avg_loss:.4f}  lr={_lr:.2e}  "
                f"elapsed={_elapsed:.1f}s",
                flush=True,
            )

        _step += 1

_train_time = time.time() - _start
print(f"\nTraining done in {_train_time:.1f}s ({_train_time/60:.1f} min)")

# ---- Loss curve 图表化 (训练次还是目视，确认) ----
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 4))
_x = np.arange(1, len(_loss_history) + 1)
_y = np.array(_loss_history)
ax.plot(_x, _y, color="#8888cc", alpha=0.4, linewidth=0.8, label="loss (raw)")

# 移動平均，噪声 smoothing (window = LOG_FREQ)
if len(_y) >= TRAIN_LOG_FREQ_PY:
    _w = TRAIN_LOG_FREQ_PY
    _smooth = np.convolve(_y, np.ones(_w) / _w, mode="valid")
    _x_smooth = _x[_w - 1:]
    ax.plot(_x_smooth, _smooth, color="#2244aa", linewidth=1.5,
            label=f"moving avg (window={_w})")

ax.set_xlabel("step")
ax.set_ylabel("Flow Matching loss (MSE)")
ax.set_title(
    f"Training loss ({len(_loss_history)} steps, "
    f"final={_loss_history[-1]:.4f}, "
    f"min={min(_loss_history):.4f})"
)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()

plt.show()

# ---- checkpoint 保存 (训练对象 module + action stats + hyperparams) ----
# flow_head 仅， vlm_projector, state_proj 也训练对象因此一緒保存到.
# (保存与恢复时随机初期化的推理走)
_ckpt = {
    "flow_head_state_dict":     smolvla.flow_head.state_dict(),
    "vlm_projector_state_dict": smolvla.vlm_projector.state_dict(),
    "state_proj_state_dict":    smolvla.state_proj.state_dict(),
    "action_mean": action_mean.cpu(),
    "action_std":  action_std.cpu(),
    "chunk_size":        smolvla.chunk_size,
    "action_dim":        smolvla.action_dim,
    "state_dim":         smolvla.state_dim,
    "hidden_dim":        smolvla.hidden_dim,
    "loss_history":      _loss_history,
}
_ckpt_path = PY_CHECKPOINT_DIR / "smolvla_ckpt.pt"
torch.save(_ckpt, _ckpt_path)
print(f"\nSaved checkpoint to {_ckpt_path}")

## 10. 评估与 rollout 视频

在 LIBERO 环境中运行策略、记录 frame，最后编码为 mp4。评估使用与训练一致的预处理和 action 后处理。


### 10.1 在 LIBERO 环境中 rollout

在 2 个任务上各运行 1 个 episode，测量成功率并收集画面。每次新 episode 开始前要重置 action queue。


In [ ]:
# ==============================================================
# 10.1 LIBERO 环境， rollout (成功率 + 帧収集)
# ==============================================================
from lerobot.envs.libero import LiberoEnv
from libero.libero import benchmark as libero_benchmark

# ---- 评估配置 (小时短縮的为了 2 task × 1 ep. 个気若 PY_EVAL_EPISODES_PER_TASK 増和) ----
PY_EVAL_TASK_IDS = [0, 1]              # LIBERO-Spatial 的 task 0-1 仅
PY_EVAL_EPISODES_PER_TASK = 1          # 每个 task 1 episode. 合计 2 rollout
PY_EVAL_MAX_STEPS = 300                # 1 episode 的最大 env step
PY_EVAL_NUM_FLOW_STEPS = 10            # Flow matching 的 Euler step 几


def rollout_one_episode(model, env, seed: int, collect_frames: bool = False) -> dict:
    """1 episode 执行并结果 (成功可否 + 可选， frame 列) 返次.

    action chunking 的仕組:
      chunk 予測 → chunk_size 个的 action 順到 env 到投 → 消費切下一次 chunk 予測
    """
    model.eval()
    device = next(model.flow_head.parameters()).device

    obs, info = env.reset(seed=seed)
    task_desc = env.task_description  # 示示示示例: "pick up the black bowl on the plate"

    chunk_buffer = None    # 当前保持并 action chunk (predicted, denormalized numpy)
    chunk_step = 0         # chunk 内，何番目到还是
    success = False
    total_steps = 0
    frames = [] if collect_frames else None

    # 1 episode 内的 env step 几进度显示 (rollout 完成，自動的到消)
    from tqdm.auto import tqdm as _tqdm_inner
    _inner_pbar = _tqdm_inner(
        total=PY_EVAL_MAX_STEPS, desc="  steps", unit="step", leave=False
    )

    while total_steps < PY_EVAL_MAX_STEPS:
        # 当前观测画像 (front 相机)
        img = obs["pixels"]["image"]   # (H, W, C) uint8

        # frame 捕获 (视频化用)
        if collect_frames:
            # LIBERO 是 agentview 上下反転并返的，、视频用是普通的向到恢复
            frames.append(img[::-1, ::-1].copy())

        # ★ 当前 robot state  dataset 的 observation.state 形式到整形.
        # 个教材，使 LIBERO 几据集，是、
        # joints.pos (7,) + gripper.qpos 的先頭要素 (1,) = 8 维度与并構成执行.
        try:
            _joint_pos = obs["robot_state"]["joints"]["pos"]            # (7,) numpy
            _gripper_q = obs["robot_state"]["gripper"]["qpos"]           # (2,) numpy
            _state_vec = np.concatenate(
                [_joint_pos, _gripper_q[:1]], axis=0
            ).astype(np.float32)                                          # (8,)
        except (KeyError, TypeError) as _e:
            # 零，埋誤魔化、原因明示并停止执行.
            # 発生時是 LiberoEnv 的 obs_type="pixels_agent_pos" 确认.
            raise RuntimeError(
                "LIBERO env observation 从 state 构建，不: "
                f"{_e}. obs.keys={list(obs.keys())}"
            ) from _e
        state_tensor = torch.from_numpy(_state_vec).unsqueeze(0).to(device)  # (1, 8)

        # chunk 空 or 使切新的 chunk 予測
        if chunk_buffer is None or chunk_step >= chunk_buffer.shape[0]:
            with torch.no_grad():
                pred_normalized = model.sample_actions(
                    [img], [task_desc], state_tensor,
                    num_steps=PY_EVAL_NUM_FLOW_STEPS,
                )  # (1, chunk_size, 7)
            # 归一化空间 → 実 action 尺度方向
            pred_denorm = denormalize_actions(pred_normalized[0])   # (chunk_size, 7)
            chunk_buffer = pred_denorm.cpu().numpy()
            chunk_step = 0

        # chunk 从下一次 1 action 取出并 env 到投
        action = chunk_buffer[chunk_step]  # (7,)
        chunk_step += 1
        total_steps += 1

        obs, reward, terminated, truncated, info = env.step(action)
        _inner_pbar.update(1)

        if terminated or truncated:
            success = bool(info.get("success", terminated))
            break

    _inner_pbar.close()
    return {
        "success": success,
        "steps": total_steps,
        "frames": frames,   # collect_frames=False 若 None
    }


# ---- 全部部 task × 全部部 episode  rollout ----
print(
    f"Evaluating custom SmolVLA on {len(PY_EVAL_TASK_IDS)} tasks × "
    f"{PY_EVAL_EPISODES_PER_TASK} episodes...",
    flush=True,
)

# LIBERO benchmark suite 获取
_suite = libero_benchmark.get_benchmark_dict()["libero_spatial"]()

py_eval_results = {"per_task": [], "overall_success": None, "rollouts": []}
_total_success = 0
_total_rollouts = 0

# 全部部 rollout 的进度- (外側)
from tqdm.auto import tqdm as _tqdm_outer
_total_expected = len(PY_EVAL_TASK_IDS) * PY_EVAL_EPISODES_PER_TASK
_outer_pbar = _tqdm_outer(total=_total_expected, desc="rollout", unit="ep")

for task_id in PY_EVAL_TASK_IDS:
    _task_success = 0
    for ep_i in range(PY_EVAL_EPISODES_PER_TASK):
        # env 毎 episode 新規创建 (状態持越)
        env = LiberoEnv(
            task_suite=_suite,
            task_id=task_id,
            task_suite_name="libero_spatial",
            episode_index=ep_i,
            n_envs=PY_EVAL_EPISODES_PER_TASK,
            observation_height=256,
            observation_width=256,
            camera_name=["agentview_image"],
            camera_name_mapping={"agentview_image": "image"},
            control_mode="relative",
            is_libero_plus=True,
            obs_type="pixels_agent_pos",   # ← state 输入用. ， obs 到 "robot_state" 入
        )
        # 最初的 episode 仅 frame 集视频 (内存节省)
        collect = (task_id == PY_EVAL_TASK_IDS[0] and ep_i == 0)
        result = rollout_one_episode(
            smolvla, env, seed=EVAL_SEED + task_id * 100 + ep_i,
            collect_frames=collect,
        )
        env.close()

        _task_success += int(result["success"])
        _total_success += int(result["success"])
        _total_rollouts += 1
        py_eval_results["rollouts"].append({
            "task_id": task_id,
            "episode": ep_i,
            "success": result["success"],
            "steps": result["steps"],
            "frames": result["frames"],
        })
        _outer_pbar.update(1)
        _outer_pbar.set_postfix_str(
            f"task {task_id} ep {ep_i}: "
            f"{'✅ SUCCESS' if result['success'] else '❌ FAIL'} "
            f"({result['steps']} steps)"
        )

    _task_rate = 100.0 * _task_success / PY_EVAL_EPISODES_PER_TASK
    py_eval_results["per_task"].append({
        "task_id": task_id,
        "task": SPATIAL_TASK_NAMES[task_id],
        "success_rate": _task_rate,
        "n_success": _task_success,
        "n_episodes": PY_EVAL_EPISODES_PER_TASK,
    })

_outer_pbar.close()

py_eval_results["overall_success"] = (
    100.0 * _total_success / max(_total_rollouts, 1)
)

print(f"\n=== Custom SmolVLA overall: {py_eval_results['overall_success']:.1f}% ===")

### 10.2 编码并嵌入视频

用 `av`（PyAV）把 frame 序列编码成 mp4，并把视频嵌入 notebook。


In [ ]:
# ==============================================================
# 10.2 frame 列 mp4 化并埋包含显示
# ==============================================================
import av
from IPython.display import Video, display

PY_VIDEO_DIR = WORKDIR / "py_videos"
PY_VIDEO_DIR.mkdir(parents=True, exist_ok=True)


def write_mp4(frames: list, path: Path, fps: int = 20) -> None:
    """RGB uint8 的 frame 列 H.264 mp4 到書出."""
    if not frames:
        raise RuntimeError("frames 空")

    h, w = frames[0].shape[:2]
    # H.264 的 yuv420p 是宽度・高度偶几，有需要有. 奇几若 1px 降低.
    h -= h % 2
    w -= w % 2

    with av.open(str(path), mode="w") as container:
        stream = container.add_stream("libx264", rate=fps)
        stream.width = w
        stream.height = h
        stream.pix_fmt = "yuv420p"
        # crf 小程度高画質 & 文条大. 23  libx264 的既定
        stream.options = {"crf": "23"}

        for fr in frames:
            fr = fr[:h, :w]                          # 偶几化的为了切詰
            frame = av.VideoFrame.from_ndarray(fr, format="rgb24")
            for packet in stream.encode(frame):
                container.mux(packet)
        # flush 残几据包
        for packet in stream.encode():
            container.mux(packet)


# rollout 结果从 frames 持也的仅 mp4 化并显示
saved = 0
for r in py_eval_results["rollouts"]:
    if not r["frames"]:
        continue
    mp4_path = PY_VIDEO_DIR / f"task{r['task_id']}_ep{r['episode']}.mp4"
    write_mp4(r["frames"], mp4_path, fps=20)
    label = "SUCCESS" if r["success"] else "FAIL"
    print(f"\n▶ task {r['task_id']} ep {r['episode']}  ({label}, {r['steps']} steps)")
    display(Video(str(mp4_path), embed=True, html_attributes="controls loop"))
    saved += 1

if saved == 0:
    print("(collect_frames=False  rollout 还是. section 10.1 ， collect 标志 true 到并请)")
else:
    print(f"\nSaved {saved} mp4(s) to {PY_VIDEO_DIR}")

## 11. 导出提交日志（基础版要求）

执行本 section 会生成 `submission.json`、辅助说明和依赖记录，并触发浏览器下载。提交时只上传 `submission.json`；Basic 版本的哈希和日志结构属于程序契约，请不要改变。


In [ ]:
# ==============================================================
# 11 提交用日志 (Basic 版个个)
# ==============================================================


import hashlib
import json
import platform
import subprocess
from datetime import datetime, timezone

# ▼ 可选: 何試完成还是自由到書残 (未記入也 OK) ▼
METHODS_NOTES = "(自由記述: 工夫試完成还是等)"

SUBMISSION_DIR = WORKDIR / "submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)


def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ---- 执行的証拠 ----
_ckpt_path = PY_CHECKPOINT_DIR / "smolvla_ckpt.pt"
fingerprints = {}
if _ckpt_path.is_file():
    fingerprints["smolvla_ckpt.pt"] = sha256_of_file(_ckpt_path)

# 视频文条的有無确认
_video_paths = sorted(
    (WORKDIR / "py_videos").glob("*.mp4")
)

execution_evidence = {
    "vlm_hidden_dim": int(vlm_encoder.hidden_dim),
    "transformer_hidden_dim": int(smolvla.transformer_hidden_dim),
    "trainable_params": int(sum(
        p.numel() for p in smolvla.parameters() if p.requires_grad
    )),
    "training_steps_completed": int(TRAIN_STEPS_PY),
    "training_final_loss": (
        float(_loss_history[-1]) if _loss_history else None
    ),
    "eval_total_rollouts": int(len(py_eval_results.get("rollouts", []))),
    "eval_success_count": int(sum(
        1 for r in py_eval_results.get("rollouts", []) if r.get("success")
    )),
    "eval_overall_success_percent": float(
        py_eval_results.get("overall_success", 0.0)
    ),
    "video_generated": bool(_video_paths),
    "video_count": len(_video_paths),
}

# ---- 环境情報 ----
try:
    pip_freeze = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"], text=True,
    )
except Exception as e:
    pip_freeze = f"# pip freeze failed: {e}"

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

# ---- 提交 JSON 構成 ----
submission = {
    "notebook_version": "basic-v1",
    "submitted_at": datetime.now(timezone.utc).isoformat(),
    "methods_notes": METHODS_NOTES,
    "todos_completed": [
        "TODO_1_time_embedding_outer_product",
        "TODO_2_flow_matching_interpolation",
        "TODO_3_target_velocity",
        "TODO_4_euler_step",
    ],
    "execution_evidence": execution_evidence,
    "model_fingerprint": fingerprints,
    "environment": environment,
    "hyperparams": {
        "chunk_size": smolvla.chunk_size,
        "action_dim": smolvla.action_dim,
        "transformer_hidden_dim": smolvla.transformer_hidden_dim,
        "num_transformer_layers": len(smolvla.flow_head.blocks),
        "train_steps": TRAIN_STEPS_PY,
        "batch_size": TRAIN_BATCH_SIZE_PY,
        "learning_rate": TRAIN_LR_PY,
        "warmup_steps": TRAIN_WARMUP_PY,
        "seed": SEED,
        "num_flow_steps_eval": PY_EVAL_NUM_FLOW_STEPS,
    },
}

# ---- 書出 ----
sub_json = SUBMISSION_DIR / "submission.json"
sub_json.write_text(
    json.dumps(submission, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# 人間可読摘要
_per_task = "\n".join(
    f"- task {r['task_id']} ep {r['episode']}: "
    f"{'✅' if r['success'] else '❌'} ({r['steps']} steps)"
    for r in py_eval_results.get("rollouts", [])
)
sub_md = SUBMISSION_DIR / "submission.md"
sub_md.write_text(
    f"""# Submission

**Submitted**: {submission['submitted_at']}
**Version**: basic-v1

## TODO 完成状況
✅ 全部部的 TODO 実装済 (notebook 这里到执行，時点，確定)

## 执行的証拠
- VLM hidden dim: {execution_evidence['vlm_hidden_dim']}
- Transformer hidden dim: {execution_evidence['transformer_hidden_dim']}
- 训练可能参数: {execution_evidence['trainable_params']:,}
- 训练 step: {execution_evidence['training_steps_completed']}
- 最終 loss: {execution_evidence['training_final_loss']}
- Rollout 几: {execution_evidence['eval_total_rollouts']}
- 成功几: {execution_evidence['eval_success_count']}
- 视频生成: {'✅' if execution_evidence['video_generated'] else '❌'}

## Rollout 詳細
{_per_task}

## Notes
{METHODS_NOTES}

## Environment
- {environment['gpu']}
- torch {environment['torch']}, cuda {environment['cuda']}
- python {environment['python']}
""",
    encoding="utf-8",
)

# pip freeze 也同梱
(SUBMISSION_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")

# ---- Drive 到备份 (runtime終了，消到) ----
# 提交物是 submission.json 仅、参照用到 md / pip_freeze 也一緒到置.
_drive_dir = DRIVE_BACKUP_DIR / "submission"
_drive_dir.mkdir(parents=True, exist_ok=True)
for f in [sub_json, sub_md, SUBMISSION_DIR / "pip_freeze.txt"]:
    shutil.copy2(f, _drive_dir / f.name)

# ---- submission.json 浏览器到下载 ----
# 経験的到、ipywidgets Button 的 on_click 経由， files.download 呼与
# 浏览器到是 download prompt 出与有.
# cell执行的 user gesture 有効、cell末尾的同期呼出的更確実.
print("✅ 提交用文条 (submission.json) 生成完成:")
print(f"  {sub_json}")
print()
print("参照用到以下也生成:")
print(f"  {sub_md}")
print(f"  {SUBMISSION_DIR / 'pip_freeze.txt'}")
print()
print(f"✅ Drive 到也已备份: {_drive_dir}")
print()

# 浏览器到 download 投. 这里，自動的到下载始.
# 也 download prompt 出、Drive 从直接获取执行的確実:
#   https://drive.google.com/ → MyDrive → smolvla_task → submission → submission.json
from google.colab import files
try:
    files.download(str(sub_json))
    print("📥 浏览器到 submission.json 的下载送信完成.")
except Exception as _e:
    print(f"⚠️ files.download 失败: {_e}")

print()
print("=" * 60)
print("💡 download prompt 出場合是 Drive 从获取并请:")
print()
print("   Drive → MyDrive → smolvla_task → submission → submission.json")
print()
print(f"   (Colab 路径: {_drive_dir / sub_json.name})")
print("=" * 60)